# 02. 소분류 접근성 지표와 이용건수 상관분석
- 목적: 생활형 소분류 5개에 대해 접근성 지표와 대상자 1인당 이용건수의 상관관계를 비교함.
- 대상 소분류: 체육시설, 영화, 사진관, 도서, 온천.
- 비교 지표: 정부 최근접 접근성, 서비스권역 인구비중, 선호 미반영 SFCA, 선호 반영 H3SFCA.

## 01. 분석 환경 설정
- 프로젝트 경로와 산출물 저장 경로를 설정함.
- 기존 접근성 산출물과 네트워크 pair를 재사용함.
- 새 대용량 중간 파일은 저장하지 않음.

In [ ]:
import gc
import heapq
import math
import warnings
from pathlib import Path

import geopandas as gpd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from scipy.spatial import cKDTree

warnings.filterwarnings("ignore", category=UserWarning)

plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False
pd.set_option("display.max_columns", 120)
pd.set_option("display.float_format", "{:,.6f}".format)

BASE_PATH = Path().resolve()

if BASE_PATH.name == "eda":
    PROJECT_PATH = BASE_PATH.parents[1]
elif BASE_PATH.name == "notebooks":
    PROJECT_PATH = BASE_PATH.parent
elif (BASE_PATH / "analysis_table").exists():
    PROJECT_PATH = BASE_PATH
else:
    PROJECT_PATH = Path(r"C:\project\oracle_mnc_project")

EDA_PATH = PROJECT_PATH / "notebooks" / "eda"
OUTPUT_PATH = EDA_PATH / "OUTPUT"
IMAGE_PATH = EDA_PATH / "IMAGE"
ACCESS_OUTPUT_PATH = PROJECT_PATH / "notebooks" / "access" / "OUTPUT"
H3_PATH = ACCESS_OUTPUT_PATH / "h3sfca"
NETWORK_OUTPUT_PATH = PROJECT_PATH / "analysis_table" / "data" / "output" / "network_competition_25km"
DATA_INPUT_PATH = PROJECT_PATH / "analysis_table" / "data" / "input"
KTDB_PATH = DATA_INPUT_PATH / "network" / "ktdb_transport_network"
MNC_CARD_PATH = PROJECT_PATH / "data" / "raw" / "mnc_card" / "mnc_seoul_usage_issuance_2021_2025.xlsx"

OUTPUT_PATH.mkdir(parents=True, exist_ok=True)
IMAGE_PATH.mkdir(parents=True, exist_ok=True)

GRID_COMPETITION_PATH = NETWORK_OUTPUT_PATH / "경쟁권25km_분석격자.parquet"
STORE_COMPETITION_PATH = NETWORK_OUTPUT_PATH / "경쟁권25km_분석가맹점.parquet"
WALK_PAIR_PATH = NETWORK_OUTPUT_PATH / "경쟁권25km_도보접근성.parquet"
TRANSIT_PAIR_PATH = NETWORK_OUTPUT_PATH / "경쟁권25km_대중교통접근성.parquet"
SUPPLY_PATH = H3_PATH / "문화누리_가맹점_카테고리별_공급량.csv"
H3_DEMAND_PATH = H3_PATH / "h3sfca_격자_중분류_선호수요.parquet"
SEOUL_GRID_POP_PATH = H3_PATH / "sfca_no_preference_격자_중분류_접근성.csv"

NODE_PATH = next(KTDB_PATH.rglob("2025node.txt"))
LINK_PATH = next(KTDB_PATH.rglob("2025link.txt"))
ANALYSIS_AREA_PATH = NETWORK_OUTPUT_PATH / "경쟁권25km_분석권역.gpkg"

SERVICE_LIMIT_M = 10_000
ROAD_BUFFER_M = 2_000
ROAD_LINK_TYPES = list(range(101, 109))
WALK_CUTOFF_M = 750.0
TRANSIT_CUTOFF_MIN = 20.0
PRIMARY_LAMBDA = 1.2

selected_subcategory = pd.DataFrame([
    {"중분류": "체육시설", "소분류": "체육시설", "접근수단": "도보", "이용건수칼럼": "체육시설\n(건)", "비고": "생활형"},
    {"중분류": "영상", "소분류": "영화", "접근수단": "도보", "이용건수칼럼": "영화\n(건)", "비고": "영상 중 영화만 사용"},
    {"중분류": "미술", "소분류": "사진관", "접근수단": "대중교통", "이용건수칼럼": "사진관\n(건)", "비고": "미술 중 사진관만 사용"},
    {"중분류": "도서", "소분류": "도서", "접근수단": "도보", "이용건수칼럼": "도서\n(건)", "비고": "생활형"},
    {"중분류": "관광지", "소분류": "온천", "접근수단": "대중교통", "이용건수칼럼": "온천\n(건)", "비고": "관광지 중 온천만 사용"},
])

print("PROJECT_PATH:", PROJECT_PATH)
print("OUTPUT_PATH:", OUTPUT_PATH)
print("IMAGE_PATH:", IMAGE_PATH)
print("분석 대상 소분류")
display(selected_subcategory)

## 02. 2025년 소분류 이용건수 타깃 생성
- 문화누리카드 2025년 구별 이용건수 원자료에서 선택 소분류 건수를 추출함.
- 이용지표는 `소분류 이용건수 / 구별 문화누리대상자 추정인구`로 계산함.
- 총량 보정용 인구는 기존 접근성 분석의 서울 100m 격자 문화누리대상자 추정인구를 구별 합산해 사용함.

In [ ]:
def clean_number(series):
    return pd.to_numeric(
        series.astype(str).str.replace(",", "", regex=False).str.strip(),
        errors="coerce"
    )

seoul_grid_base = pd.read_csv(
    SEOUL_GRID_POP_PATH,
    encoding="utf-8-sig",
    usecols=["GRID_CD", "시군구", "행정동", "중심점_x", "중심점_y", "추정_인구수", "문화누리대상자_추정_인구수"]
).drop_duplicates("GRID_CD").copy()

for col in ["추정_인구수", "문화누리대상자_추정_인구수"]:
    seoul_grid_base[col] = pd.to_numeric(seoul_grid_base[col], errors="coerce").fillna(0)

gu_mnc_pop = (
    seoul_grid_base
    .groupby("시군구", as_index=False)
    .agg(
        구별_총추정인구=("추정_인구수", "sum"),
        구별_문화누리대상자추정인구=("문화누리대상자_추정_인구수", "sum"),
        서울격자수=("GRID_CD", "nunique")
    )
)

usage_raw = pd.read_excel(MNC_CARD_PATH, sheet_name="2025")
usage_raw["광역"] = usage_raw["광역"].astype(str).str.strip()
usage_raw["기초"] = usage_raw["기초"].astype(str).str.strip()
usage_raw = usage_raw[usage_raw["광역"].eq("서울")].copy()
usage_raw = usage_raw.rename(columns={"기초": "시군구"})
usage_raw["시군구"] = usage_raw["시군구"].astype(str).str.strip()

usage_list = []
for _, row in selected_subcategory.iterrows():
    col = row["이용건수칼럼"]
    temp = usage_raw[["시군구", col]].copy()
    temp = temp.rename(columns={col: "이용건수"})
    temp["중분류"] = row["중분류"]
    temp["소분류"] = row["소분류"]
    usage_list.append(temp)

usage = pd.concat(usage_list, ignore_index=True)
usage["이용건수"] = clean_number(usage["이용건수"]).fillna(0)
usage = usage.merge(gu_mnc_pop, on="시군구", how="left")
usage["대상자1인당_이용건수"] = np.where(
    usage["구별_문화누리대상자추정인구"] > 0,
    usage["이용건수"] / usage["구별_문화누리대상자추정인구"],
    np.nan
)
usage["대상자천명당_이용건수"] = usage["대상자1인당_이용건수"] * 1000

print("소분류 이용건수 타깃 구조:", usage.shape)
print("시군구 수:", usage["시군구"].nunique())
print("소분류 수:", usage["소분류"].nunique())
print("문화누리대상자 인구 결측:", usage["구별_문화누리대상자추정인구"].isna().sum())
print("이용건수 결측:", usage["이용건수"].isna().sum())

print("\n소분류별 이용건수 요약")
display(
    usage.groupby(["중분류", "소분류"], as_index=False)
    .agg(
        이용건수합=("이용건수", "sum"),
        대상자천명당_이용건수_평균=("대상자천명당_이용건수", "mean"),
        이용건수최대=("이용건수", "max")
    )
)

display(usage.head())

## 03. 가맹점·공급량·수요 데이터 로드
- 선택 소분류에 해당하는 문화누리 가맹점만 추출함.
- 공급량은 기존 H3SFCA에서 구축한 가맹점별 공급량을 그대로 사용함.
- H3SFCA 선호수요는 ML 결과가 중분류 기준이므로, 해당 소분류의 상위 중분류 선호수요를 사용함.

In [ ]:
grid_all = pd.read_parquet(
    GRID_COMPETITION_PATH,
    columns=["GRID_CD", "시도", "서울여부", "시군구", "행정동", "중심점_x", "중심점_y", "추정_인구수"]
)
grid_all["추정_인구수"] = pd.to_numeric(grid_all["추정_인구수"], errors="coerce").fillna(0)

supply = pd.read_csv(SUPPLY_PATH, encoding="utf-8-sig")
supply = supply.merge(
    selected_subcategory[["중분류", "소분류", "접근수단"]],
    on=["중분류", "소분류"],
    how="inner"
)
supply = supply[supply["분석포함"].eq(True)].copy()
supply["공급량"] = pd.to_numeric(supply["공급량"], errors="coerce").fillna(0)
supply["소분류분석명"] = supply["소분류"]

h3_demand = pd.read_parquet(H3_DEMAND_PATH)
h3_demand = h3_demand[h3_demand["중분류"].isin(selected_subcategory["중분류"].unique())].copy()
h3_demand["선호수요"] = (
    pd.to_numeric(h3_demand["일반인구_선호수요"], errors="coerce").fillna(0)
    + PRIMARY_LAMBDA * pd.to_numeric(h3_demand["문화누리대상자_기본선호수요"], errors="coerce").fillna(0)
)

print("전체 격자 구조:", grid_all.shape)
print("서울 결과 격자 수:", grid_all["서울여부"].sum())
print("선택 소분류 공급 가맹점 구조:", supply.shape)
print("H3 선호수요 구조:", h3_demand.shape)
print("공급량 0 이하:", (supply["공급량"] <= 0).sum())
print("가맹점_ID 중복:", supply["가맹점_ID"].duplicated().sum())

print("\n선택 소분류별 가맹점 수와 공급량")
display(
    supply.groupby(["중분류", "소분류", "접근수단"], as_index=False)
    .agg(
        가맹점수=("가맹점_ID", "nunique"),
        공급량합=("공급량", "sum"),
        공급량평균=("공급량", "mean")
    )
)

display(supply.head())

## 04. 정부 최근접 접근성 및 서비스권역 인구비중 계산
- 정부식 최근접 접근성은 기존 공공기관식 분석과 동일하게 KTDB 도로망 최단거리로 계산함.
- 상관분석 지표값은 요청대로 `-문화시설_접근거리_m` 원값을 사용함.
- 인구비중은 10km 안에 해당 소분류 가맹점이 1개 이상 있는 격자의 문화누리대상자 비중으로 계산함.

In [ ]:
def build_road_graph():
    node = pd.read_csv(
        NODE_PATH,
        sep=r"\s+",
        skiprows=1,
        header=None,
        names=["record_type", "node_id", "x", "y"],
        engine="python"
    ).drop(columns="record_type")

    link = pd.read_csv(
        LINK_PATH,
        sep=r"\s+",
        skiprows=1,
        header=None,
        names=[
            "record_type", "from_node", "to_node", "length_km", "mode_code",
            "link_type", "lane", "capacity", "speed", "vdf", "cost"
        ],
        engine="python"
    ).drop(columns="record_type")

    node["node_id"] = pd.to_numeric(node["node_id"], errors="coerce")
    node["x"] = pd.to_numeric(node["x"], errors="coerce")
    node["y"] = pd.to_numeric(node["y"], errors="coerce")
    node = node.dropna(subset=["node_id", "x", "y"]).copy()
    node["node_id"] = node["node_id"].astype("int64")

    for col in ["from_node", "to_node", "length_km", "link_type"]:
        link[col] = pd.to_numeric(link[col], errors="coerce")

    link = link.dropna(subset=["from_node", "to_node", "length_km", "link_type"]).copy()
    link["from_node"] = link["from_node"].astype("int64")
    link["to_node"] = link["to_node"].astype("int64")
    link["link_type"] = link["link_type"].astype("int64")
    link["length_m"] = link["length_km"] * 1000

    ktdb_crs = (
        "+proj=tmerc +lat_0=38 +lon_0=128 +k=0.9999 "
        "+x_0=400000 +y_0=600000 +ellps=bessel "
        "+towgs84=-146.43,507.89,681.46 +units=m +no_defs"
    )

    node_gdf = gpd.GeoDataFrame(
        node.copy(),
        geometry=gpd.points_from_xy(node["x"], node["y"]),
        crs=ktdb_crs
    ).to_crs("EPSG:5179")

    analysis_area = gpd.read_file(ANALYSIS_AREA_PATH).to_crs("EPSG:5179").dissolve()[["geometry"]].reset_index(drop=True)
    analysis_area_buffer = analysis_area.copy()
    analysis_area_buffer["geometry"] = analysis_area_buffer.geometry.buffer(ROAD_BUFFER_M)

    node_in_buffer = gpd.sjoin(
        node_gdf,
        analysis_area_buffer[["geometry"]],
        how="inner",
        predicate="within"
    ).drop(columns="index_right", errors="ignore")

    link_road = link[
        (link["length_m"] > 0) &
        (link["link_type"].isin(ROAD_LINK_TYPES))
    ].copy()

    buffer_node_ids = set(node_in_buffer["node_id"])
    link_road_area = link_road[
        link_road["from_node"].isin(buffer_node_ids) |
        link_road["to_node"].isin(buffer_node_ids)
    ].copy()

    road_node_ids = set(link_road_area["from_node"]) | set(link_road_area["to_node"])
    node_road_area = node_gdf[node_gdf["node_id"].isin(road_node_ids)].copy()

    edge = (
        link_road_area[["from_node", "to_node", "length_m"]]
        .groupby(["from_node", "to_node"], as_index=False)["length_m"]
        .min()
    )

    graph = nx.Graph()
    graph.add_weighted_edges_from(edge[["from_node", "to_node", "length_m"]].itertuples(index=False, name=None))

    return graph, node_road_area


def nearest_facility_dijkstra(graph, source_cost, cutoff):
    dist = {}
    nearest_node = {}
    heap = []

    for node_id, cost in source_cost.items():
        if pd.isna(cost):
            continue
        if node_id not in graph:
            continue
        cost = float(cost)
        if cost < dist.get(node_id, np.inf):
            dist[node_id] = cost
            nearest_node[node_id] = node_id
            heapq.heappush(heap, (cost, node_id, node_id))

    while heap:
        cost, node_id, source_node = heapq.heappop(heap)
        if cost != dist.get(node_id):
            continue
        if cost > cutoff:
            continue

        for next_node, edge_data in graph[node_id].items():
            next_cost = cost + edge_data.get("weight", 0)
            if next_cost < dist.get(next_node, np.inf) and next_cost <= cutoff:
                dist[next_node] = next_cost
                nearest_node[next_node] = source_node
                heapq.heappush(heap, (next_cost, next_node, source_node))

    return dist, nearest_node


road_graph, road_node = build_road_graph()
node_xy = np.column_stack([road_node.geometry.x.to_numpy(), road_node.geometry.y.to_numpy()])
node_id_array = road_node["node_id"].to_numpy()
node_tree = cKDTree(node_xy)

public_grid = gpd.read_parquet(GRID_COMPETITION_PATH).to_crs("EPSG:5179")
public_grid = public_grid[public_grid["서울여부"] == True].copy()
public_grid = public_grid[["GRID_CD", "시군구", "행정동", "추정_인구수", "geometry"]].copy()
public_grid = public_grid.merge(
    seoul_grid_base[["GRID_CD", "문화누리대상자_추정_인구수"]],
    on="GRID_CD",
    how="left"
)
public_grid["문화누리대상자_추정_인구수"] = pd.to_numeric(
    public_grid["문화누리대상자_추정_인구수"],
    errors="coerce"
).fillna(0)

store_gdf = gpd.read_parquet(STORE_COMPETITION_PATH).to_crs("EPSG:5179")
public_store = store_gdf.merge(
    selected_subcategory[["중분류", "소분류"]],
    on=["중분류", "소분류"],
    how="inner"
).copy()
public_store = public_store.dropna(subset=["geometry"]).copy()

# 격자 스냅
grid_xy = np.column_stack([public_grid.geometry.centroid.x.to_numpy(), public_grid.geometry.centroid.y.to_numpy()])
grid_distance, grid_nearest_pos = node_tree.query(grid_xy, k=1)
public_grid["도로망_노드ID"] = node_id_array[grid_nearest_pos]
public_grid["격자_스냅거리_m"] = grid_distance

# 시설 스냅
store_xy = np.column_stack([public_store.geometry.x.to_numpy(), public_store.geometry.y.to_numpy()])
store_distance, store_nearest_pos = node_tree.query(store_xy, k=1)
public_store["도로망_노드ID"] = node_id_array[store_nearest_pos]
public_store["시설_스냅거리_m"] = store_distance

public_result_list = []

for _, sub_row in selected_subcategory.iterrows():
    mid = sub_row["중분류"]
    sub = sub_row["소분류"]

    category_store = public_store[(public_store["중분류"] == mid) & (public_store["소분류"] == sub)].copy()

    category_node_best = (
        category_store
        .sort_values(["도로망_노드ID", "시설_스냅거리_m", "가맹점_ID"])
        .drop_duplicates("도로망_노드ID")
        .set_index("도로망_노드ID")
    )

    source_cost = category_node_best["시설_스냅거리_m"].to_dict()
    node_cost, nearest_node = nearest_facility_dijkstra(road_graph, source_cost, SERVICE_LIMIT_M)

    temp = public_grid[["GRID_CD", "시군구", "행정동", "문화누리대상자_추정_인구수", "도로망_노드ID", "격자_스냅거리_m"]].copy()
    temp["중분류"] = mid
    temp["소분류"] = sub
    temp["노드_시설접근비용_m"] = temp["도로망_노드ID"].map(node_cost)
    temp["문화시설_접근거리_m"] = temp["격자_스냅거리_m"] + temp["노드_시설접근비용_m"]
    temp["문화시설_10km_접근가능"] = temp["문화시설_접근거리_m"] <= SERVICE_LIMIT_M
    temp["시설수"] = len(category_store)

    public_result_list.append(temp)
    print(f"{sub}: 시설 {len(category_store):,}개, 접근거리 결측 {temp['문화시설_접근거리_m'].isna().sum():,}개, 10km 접근가능 격자 {int(temp['문화시설_10km_접근가능'].sum()):,}개")

public_grid_access = pd.concat(public_result_list, ignore_index=True)

public_gu = (
    public_grid_access
    .groupby(["시군구", "중분류", "소분류"], as_index=False)
    .apply(
        lambda x: pd.Series({
            "정부최근접_가중평균거리_m": np.average(
                x.loc[x["문화시설_접근거리_m"].notna(), "문화시설_접근거리_m"],
                weights=x.loc[x["문화시설_접근거리_m"].notna(), "문화누리대상자_추정_인구수"]
            ) if x.loc[x["문화시설_접근거리_m"].notna(), "문화누리대상자_추정_인구수"].sum() > 0 else x["문화시설_접근거리_m"].mean(),
            "서비스권역_인구비중": (
                x.loc[x["문화시설_10km_접근가능"], "문화누리대상자_추정_인구수"].sum()
                / x["문화누리대상자_추정_인구수"].sum()
            ) if x["문화누리대상자_추정_인구수"].sum() > 0 else np.nan,
            "서비스권역_도달대상자수": x.loc[x["문화시설_10km_접근가능"], "문화누리대상자_추정_인구수"].sum(),
            "대상자수": x["문화누리대상자_추정_인구수"].sum(),
            "거리결측격자수": x["문화시설_접근거리_m"].isna().sum(),
            "시설수": x["시설수"].max(),
        }),
        include_groups=False
    )
)

print("정부 최근접/인구비중 구-소분류 구조:", public_gu.shape)
display(public_gu.head())

del road_graph, road_node, public_grid, public_store, public_grid_access
gc.collect()

## 05. 소분류 SFCA/H3SFCA 계산
- 선호 미반영 SFCA는 기존과 동일하게 총 추정인구를 경쟁수요로 사용함.
- 선호 반영 H3SFCA는 기존과 동일하게 공급량과 거리감쇠로 Huff 확률을 계산함.
- 거리감쇠는 기존 기본안인 구간형 감쇠, 문화누리대상자 수요가중치는 1.2를 사용함.

In [ ]:
def apply_decay(cost, mode):
    cost = pd.to_numeric(cost, errors="coerce")
    mode = pd.Series(mode, index=cost.index)
    weight = np.zeros(len(cost), dtype="float32")

    walk_idx = mode.eq("도보")
    transit_idx = mode.eq("대중교통")

    weight[walk_idx & (cost >= 0) & (cost <= 250)] = 1.0
    weight[walk_idx & (cost > 250) & (cost <= 500)] = 0.6
    weight[walk_idx & (cost > 500) & (cost <= 750)] = 0.25
    weight[transit_idx & (cost >= 0) & (cost <= 7)] = 1.0
    weight[transit_idx & (cost > 7) & (cost <= 14)] = 0.6
    weight[transit_idx & (cost > 14) & (cost <= 20)] = 0.25

    return weight


def iter_selected_pairs(pair_path, store_info, mode_name, batch_size=1_000_000):
    mode_store = store_info[store_info["접근수단"] == mode_name].copy()
    if mode_store.empty:
        return

    store_ids = set(mode_store["가맹점_ID"])
    parquet_file = pq.ParquetFile(pair_path)

    for batch in parquet_file.iter_batches(columns=["GRID_CD", "가맹점_ID", "접근수단", "접근비용"], batch_size=batch_size):
        pair = batch.to_pandas()
        pair = pair[pair["가맹점_ID"].isin(store_ids)].copy()
        if pair.empty:
            continue
        pair = pair.merge(
            mode_store[["가맹점_ID", "중분류", "소분류", "소분류분석명", "공급량"]],
            on="가맹점_ID",
            how="inner"
        )
        yield pair


store_info = supply[["가맹점_ID", "중분류", "소분류", "소분류분석명", "접근수단", "공급량"]].drop_duplicates().copy()
grid_demand = grid_all[["GRID_CD", "추정_인구수", "서울여부"]].copy()

pair_sources = [
    ("도보", WALK_PAIR_PATH),
    ("대중교통", TRANSIT_PAIR_PATH),
]

# 1차: SFCA 시설별 가중수요
sfca_facility_demand_parts = []
sfca_pair_qc = []

for mode_name, pair_path in pair_sources:
    for pair in iter_selected_pairs(pair_path, store_info, mode_name):
        pair["거리감쇠"] = apply_decay(pair["접근비용"], pair["접근수단"])
        pair = pair[pair["거리감쇠"] > 0].copy()
        pair = pair.merge(grid_demand[["GRID_CD", "추정_인구수"]], on="GRID_CD", how="left")
        pair["가중수요"] = pd.to_numeric(pair["추정_인구수"], errors="coerce").fillna(0) * pair["거리감쇠"]
        sfca_facility_demand_parts.append(
            pair.groupby(["가맹점_ID", "소분류분석명"], as_index=False).agg(
                공급량=("공급량", "first"),
                가중수요=("가중수요", "sum")
            )
        )
        sfca_pair_qc.append({"접근수단": mode_name, "pair행수": len(pair)})
        del pair
        gc.collect()

sfca_facility = (
    pd.concat(sfca_facility_demand_parts, ignore_index=True)
    .groupby(["가맹점_ID", "소분류분석명"], as_index=False)
    .agg(공급량=("공급량", "first"), 가중수요=("가중수요", "sum"))
)
sfca_facility["공급수요비"] = np.where(sfca_facility["가중수요"] > 0, sfca_facility["공급량"] / sfca_facility["가중수요"], 0)

# 2차: SFCA 격자별 접근성
sfca_access_parts = []

for mode_name, pair_path in pair_sources:
    for pair in iter_selected_pairs(pair_path, store_info, mode_name):
        pair["거리감쇠"] = apply_decay(pair["접근비용"], pair["접근수단"])
        pair = pair[pair["거리감쇠"] > 0].copy()
        pair = pair.merge(sfca_facility[["가맹점_ID", "소분류분석명", "공급수요비"]], on=["가맹점_ID", "소분류분석명"], how="left")
        pair["접근성기여"] = pair["공급수요비"].fillna(0) * pair["거리감쇠"]
        pair = pair.merge(grid_demand[["GRID_CD", "서울여부"]], on="GRID_CD", how="left")
        pair = pair[pair["서울여부"] == True].copy()
        sfca_access_parts.append(
            pair.groupby(["GRID_CD", "중분류", "소분류"], as_index=False).agg(
                SFCA_선호미반영=("접근성기여", "sum"),
                SFCA_접근가능가맹점수=("가맹점_ID", "nunique"),
                SFCA_평균접근비용=("접근비용", "mean")
            )
        )
        del pair
        gc.collect()

sfca_grid = (
    pd.concat(sfca_access_parts, ignore_index=True)
    .groupby(["GRID_CD", "중분류", "소분류"], as_index=False)
    .agg(
        SFCA_선호미반영=("SFCA_선호미반영", "sum"),
        SFCA_접근가능가맹점수=("SFCA_접근가능가맹점수", "sum"),
        SFCA_평균접근비용=("SFCA_평균접근비용", "mean")
    )
)

print("SFCA 시설 공급수요비 구조:", sfca_facility.shape)
print("SFCA 격자 접근성 구조:", sfca_grid.shape)
print("SFCA pair QC:")
display(pd.DataFrame(sfca_pair_qc).groupby("접근수단", as_index=False)["pair행수"].sum())

# H3SFCA 1차: 격자-소분류별 Huff 분모
hden_parts = []

for mode_name, pair_path in pair_sources:
    for pair in iter_selected_pairs(pair_path, store_info, mode_name):
        pair["거리감쇠"] = apply_decay(pair["접근비용"], pair["접근수단"])
        pair = pair[pair["거리감쇠"] > 0].copy()
        pair["매력도"] = pair["공급량"] * pair["거리감쇠"]
        hden_parts.append(
            pair.groupby(["GRID_CD", "소분류분석명"], as_index=False)["매력도"].sum()
            .rename(columns={"매력도": "Huff분모"})
        )
        del pair
        gc.collect()

hden = (
    pd.concat(hden_parts, ignore_index=True)
    .groupby(["GRID_CD", "소분류분석명"], as_index=False)["Huff분모"]
    .sum()
)

# H3SFCA 2차: 시설별 선호가중 수요
h3_facility_demand_parts = []

for mode_name, pair_path in pair_sources:
    for pair in iter_selected_pairs(pair_path, store_info, mode_name):
        pair["거리감쇠"] = apply_decay(pair["접근비용"], pair["접근수단"])
        pair = pair[pair["거리감쇠"] > 0].copy()
        pair["매력도"] = pair["공급량"] * pair["거리감쇠"]
        pair = pair.merge(hden, on=["GRID_CD", "소분류분석명"], how="left")
        pair["Huff확률"] = np.where(pair["Huff분모"] > 0, pair["매력도"] / pair["Huff분모"], 0)
        pair = pair.merge(h3_demand[["GRID_CD", "중분류", "선호수요"]], on=["GRID_CD", "중분류"], how="left")
        pair["가중수요"] = pair["선호수요"].fillna(0) * pair["Huff확률"]
        h3_facility_demand_parts.append(
            pair.groupby(["가맹점_ID", "소분류분석명"], as_index=False).agg(
                공급량=("공급량", "first"),
                가중수요=("가중수요", "sum")
            )
        )
        del pair
        gc.collect()

h3_facility = (
    pd.concat(h3_facility_demand_parts, ignore_index=True)
    .groupby(["가맹점_ID", "소분류분석명"], as_index=False)
    .agg(공급량=("공급량", "first"), 가중수요=("가중수요", "sum"))
)
h3_facility["공급수요비"] = np.where(h3_facility["가중수요"] > 0, h3_facility["공급량"] / h3_facility["가중수요"], 0)

# H3SFCA 3차: 격자별 접근성
h3_access_parts = []

for mode_name, pair_path in pair_sources:
    for pair in iter_selected_pairs(pair_path, store_info, mode_name):
        pair["거리감쇠"] = apply_decay(pair["접근비용"], pair["접근수단"])
        pair = pair[pair["거리감쇠"] > 0].copy()
        pair["매력도"] = pair["공급량"] * pair["거리감쇠"]
        pair = pair.merge(hden, on=["GRID_CD", "소분류분석명"], how="left")
        pair["Huff확률"] = np.where(pair["Huff분모"] > 0, pair["매력도"] / pair["Huff분모"], 0)
        pair = pair.merge(h3_facility[["가맹점_ID", "소분류분석명", "공급수요비"]], on=["가맹점_ID", "소분류분석명"], how="left")
        pair["접근성기여"] = pair["공급수요비"].fillna(0) * pair["Huff확률"]
        pair = pair.merge(h3_demand[["GRID_CD", "중분류", "선호수요"]], on=["GRID_CD", "중분류"], how="left")
        pair = pair.merge(grid_demand[["GRID_CD", "서울여부"]], on="GRID_CD", how="left")
        pair = pair[pair["서울여부"] == True].copy()
        h3_access_parts.append(
            pair.groupby(["GRID_CD", "중분류", "소분류"], as_index=False).agg(
                H3SFCA_선호반영=("접근성기여", "sum"),
                H3SFCA_접근가능가맹점수=("가맹점_ID", "nunique"),
                H3SFCA_평균접근비용=("접근비용", "mean"),
                선호수요=("선호수요", "first")
            )
        )
        del pair
        gc.collect()

h3_grid = (
    pd.concat(h3_access_parts, ignore_index=True)
    .groupby(["GRID_CD", "중분류", "소분류"], as_index=False)
    .agg(
        H3SFCA_선호반영=("H3SFCA_선호반영", "sum"),
        H3SFCA_접근가능가맹점수=("H3SFCA_접근가능가맹점수", "sum"),
        H3SFCA_평균접근비용=("H3SFCA_평균접근비용", "mean"),
        선호수요=("선호수요", "first")
    )
)

print("H3 Huff 분모 구조:", hden.shape)
print("H3 시설 공급수요비 구조:", h3_facility.shape)
print("H3 격자 접근성 구조:", h3_grid.shape)

del hden, sfca_facility, h3_facility
gc.collect()

## 06. 구-소분류 단위 지표 집계
- 접근성 지표는 서울 격자 기준으로 구-소분류 단위 문화누리대상자 가중평균을 계산함.
- 정부 최근접 접근성은 `-거리`로 방향을 맞춤.
- SFCA/H3SFCA는 원 접근성지수를 그대로 사용함.

In [ ]:
def weighted_mean(value, weight):
    value = pd.to_numeric(value, errors="coerce")
    weight = pd.to_numeric(weight, errors="coerce").fillna(0)
    valid = value.notna()

    if valid.sum() == 0:
        return np.nan

    value = value[valid]
    weight = weight[valid]

    if weight.sum() > 0:
        return np.average(value, weights=weight)
    return value.mean()


def aggregate_grid_access(access_df, value_col, out_col):
    full = seoul_grid_base[["GRID_CD", "시군구", "문화누리대상자_추정_인구수"]].merge(
        selected_subcategory[["중분류", "소분류"]],
        how="cross"
    )
    full = full.merge(access_df[["GRID_CD", "중분류", "소분류", value_col]], on=["GRID_CD", "중분류", "소분류"], how="left")
    full[value_col] = pd.to_numeric(full[value_col], errors="coerce").fillna(0)

    gu = (
        full
        .groupby(["시군구", "중분류", "소분류"], as_index=False)
        .apply(
            lambda x: pd.Series({
                out_col: weighted_mean(x[value_col], x["문화누리대상자_추정_인구수"]),
                "대상자수": x["문화누리대상자_추정_인구수"].sum(),
                "격자수": len(x),
            }),
            include_groups=False
        )
    )
    return gu

sfca_gu = aggregate_grid_access(sfca_grid, "SFCA_선호미반영", "SFCA_선호미반영")
h3_gu = aggregate_grid_access(h3_grid, "H3SFCA_선호반영", "H3SFCA_선호반영")

indicator_wide = usage[["시군구", "중분류", "소분류", "이용건수", "대상자1인당_이용건수", "대상자천명당_이용건수", "구별_문화누리대상자추정인구"]].copy()
indicator_wide = indicator_wide.merge(
    public_gu[["시군구", "중분류", "소분류", "정부최근접_가중평균거리_m", "서비스권역_인구비중", "시설수"]],
    on=["시군구", "중분류", "소분류"],
    how="left"
)
indicator_wide = indicator_wide.merge(
    sfca_gu[["시군구", "중분류", "소분류", "SFCA_선호미반영"]],
    on=["시군구", "중분류", "소분류"],
    how="left"
)
indicator_wide = indicator_wide.merge(
    h3_gu[["시군구", "중분류", "소분류", "H3SFCA_선호반영"]],
    on=["시군구", "중분류", "소분류"],
    how="left"
)

indicator_wide["정부최근접접근성_음거리"] = -indicator_wide["정부최근접_가중평균거리_m"]

indicator_long = indicator_wide.melt(
    id_vars=["시군구", "중분류", "소분류", "이용건수", "대상자1인당_이용건수", "대상자천명당_이용건수", "구별_문화누리대상자추정인구", "시설수"],
    value_vars=["정부최근접접근성_음거리", "서비스권역_인구비중", "SFCA_선호미반영", "H3SFCA_선호반영"],
    var_name="지표명",
    value_name="지표값"
)

indicator_wide.to_csv(OUTPUT_PATH / "subcategory_access_usage_gu_wide_2025.csv", index=False, encoding="utf-8-sig")
indicator_long.to_csv(OUTPUT_PATH / "subcategory_access_usage_gu_long_2025.csv", index=False, encoding="utf-8-sig")

print("구-소분류 wide 구조:", indicator_wide.shape)
print("구-소분류 long 구조:", indicator_long.shape)
print("wide 결측")
print(indicator_wide.isna().sum())
print("long 중복:", indicator_long[["시군구", "중분류", "소분류", "지표명"]].duplicated().sum())

display(indicator_wide.head())

## 07. 소분류별 상관계수 계산
- 각 소분류별로 25개 자치구의 접근성 지표와 대상자 1인당 이용건수 상관계수를 계산함.
- Pearson은 선형 관계, Spearman은 순위 관계를 확인함.
- 회귀분석은 수행하지 않음.

In [ ]:
def safe_corr(df, x_col, y_col, method):
    temp = df[[x_col, y_col]].replace([np.inf, -np.inf], np.nan).dropna()

    if len(temp) < 3:
        return np.nan
    if temp[x_col].nunique() < 2 or temp[y_col].nunique() < 2:
        return np.nan

    return temp[x_col].corr(temp[y_col], method=method)

corr_list = []

for (indicator_name, mid, sub), temp in indicator_long.groupby(["지표명", "중분류", "소분류"]):
    corr_list.append({
        "지표명": indicator_name,
        "중분류": mid,
        "소분류": sub,
        "n": temp[["지표값", "대상자1인당_이용건수"]].replace([np.inf, -np.inf], np.nan).dropna().shape[0],
        "Pearson": safe_corr(temp, "지표값", "대상자1인당_이용건수", "pearson"),
        "Spearman": safe_corr(temp, "지표값", "대상자1인당_이용건수", "spearman"),
        "지표값_평균": temp["지표값"].mean(),
        "대상자천명당_이용건수_평균": temp["대상자천명당_이용건수"].mean(),
    })

corr = pd.DataFrame(corr_list)

summary = (
    corr
    .groupby("지표명", as_index=False)
    .agg(
        소분류수=("소분류", "nunique"),
        Pearson_평균=("Pearson", "mean"),
        Pearson_중앙값=("Pearson", "median"),
        Pearson_양수소분류수=("Pearson", lambda x: (x > 0).sum()),
        Spearman_평균=("Spearman", "mean"),
        Spearman_중앙값=("Spearman", "median"),
        Spearman_양수소분류수=("Spearman", lambda x: (x > 0).sum()),
    )
)

corr.to_csv(OUTPUT_PATH / "subcategory_access_usage_correlation_2025.csv", index=False, encoding="utf-8-sig")
summary.to_csv(OUTPUT_PATH / "subcategory_access_usage_correlation_summary_2025.csv", index=False, encoding="utf-8-sig")

print("소분류별 상관계수")
display(corr.sort_values(["소분류", "지표명"]).round(4))

print("\n지표별 요약")
display(summary.round(4))

## 08. 상관계수 시각화
- 소분류별 Pearson·Spearman 상관계수를 지표별로 비교함.
- 그래프는 EDA 이미지 폴더에 저장함.
- 색상은 기존 프로젝트 톤과 유사한 오렌지·연어·갈색 계열을 사용함.

In [ ]:
indicator_order = ["정부최근접접근성_음거리", "서비스권역_인구비중", "SFCA_선호미반영", "H3SFCA_선호반영"]
indicator_label = {
    "정부최근접접근성_음거리": "정부 최근접(-거리)",
    "서비스권역_인구비중": "인구비중",
    "SFCA_선호미반영": "선호 미반영 SFCA",
    "H3SFCA_선호반영": "선호 반영 H3SFCA",
}
indicator_color = {
    "정부최근접접근성_음거리": "#8b1e16",
    "서비스권역_인구비중": "#f0a23a",
    "SFCA_선호미반영": "#ea6b2d",
    "H3SFCA_선호반영": "#168f86",
}
subcategory_order = ["체육시설", "영화", "사진관", "도서", "온천"]

for metric in ["Spearman", "Pearson"]:
    pivot = corr.pivot(index="소분류", columns="지표명", values=metric).reindex(subcategory_order)

    fig, ax = plt.subplots(figsize=(10.5, 5.5))
    x = np.arange(len(subcategory_order))
    width = 0.19

    for idx, indicator_name in enumerate(indicator_order):
        ax.bar(
            x + (idx - 1.5) * width,
            pivot[indicator_name],
            width,
            label=indicator_label[indicator_name],
            color=indicator_color[indicator_name]
        )

    ax.axhline(0, color="#333333", linewidth=0.9)
    ax.set_xticks(x)
    ax.set_xticklabels(subcategory_order)
    ax.set_ylabel(f"{metric} 상관계수")
    ax.set_title(f"소분류 접근성 지표와 대상자 1인당 이용건수 {metric} 상관")
    ax.legend(frameon=False, ncol=2)
    ax.grid(axis="y", alpha=0.25)

    out_path = IMAGE_PATH / f"subcategory_access_usage_{metric.lower()}_2025.png"
    plt.savefig(out_path, dpi=220, bbox_inches="tight")
    plt.close()
    print("저장:", out_path)

print("\n저장된 산출물")
for path in sorted(OUTPUT_PATH.glob("subcategory_access_usage*.csv")):
    print("-", path.name)

print("\n저장된 이미지")
for path in sorted(IMAGE_PATH.glob("subcategory_access_usage*.png")):
    print("-", path.name)

## 09. 결과 해석용 요약
- 절대 상관계수 기준으로 소분류별 가장 크게 움직인 지표를 확인함.
- 양의 상관은 접근성 지표가 높을수록 대상자 1인당 이용건수가 높은 방향임.
- 음의 상관은 접근성이 높은 구에서 이용건수가 낮은 방향으로 움직인다는 뜻이며, 인과관계로 해석하지 않음.

In [ ]:
best_abs = (
    corr
    .assign(abs_spearman=lambda x: x["Spearman"].abs())
    .sort_values(["소분류", "abs_spearman"], ascending=[True, False])
    .groupby("소분류", as_index=False)
    .head(1)
    .drop(columns="abs_spearman")
)

print("소분류별 절대 Spearman 기준 가장 큰 지표")
display(best_abs[["중분류", "소분류", "지표명", "Pearson", "Spearman"]].round(4))

print("\n양의 Spearman 상관만 보기")
display(
    corr[corr["Spearman"] > 0]
    .sort_values("Spearman", ascending=False)
    [["중분류", "소분류", "지표명", "Pearson", "Spearman"]]
    .round(4)
)

print("\n분석상 주의")
print("- 영화, 사진관, 온천은 중분류 산출물을 그대로 쓴 것이 아니라 해당 소분류 가맹점만 필터해 새로 계산했다.")
print("- H3SFCA 선호수요는 ML 결과가 중분류 기준이므로 소분류별 선호확률이 아니라 상위 중분류 선호수요를 사용했다.")
print("- 정부 최근접 접근성은 요청대로 -거리 원값을 사용했다.")
print("- 회귀분석은 수행하지 않았다.")

## 10. 생활인접권 선택시설 / 특수목적시설 소분류별 정정 분석
- 생활인접권 선택시설 7개 소분류와 특수목적시설군 11개 소분류를 모두 분리해 계산함.
- 선택 소분류: 체육시설, 영화, 사진관, 도서, 온천, 음악, 체육용품.
- 특수목적시설군은 선택 7개 소분류에 해당하는 가맹점 ID를 제외한 뒤 남은 가맹점을 소분류별로 계산함.
- 음악·체육용품은 ML 선호확률이 없어 H3SFCA 칼럼에 선호 미반영 SFCA 값을 대체함.
- 상관분석은 시군구별 대상자 1인당 이용건수와 접근성 지표 간 Pearson·Spearman으로 계산함.

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display, Image

EDA_PATH = Path(r'C:\project\oracle_mnc_project\notebooks\eda')
OUTPUT_PATH = EDA_PATH / 'OUTPUT'
IMAGE_PATH = EDA_PATH / 'IMAGE'

spearman_table = pd.read_csv(OUTPUT_PATH / 'subcategory_access_usage_spearman_table_2025.csv', encoding='utf-8-sig')
pearson_table = pd.read_csv(OUTPUT_PATH / 'subcategory_access_usage_pearson_table_2025.csv', encoding='utf-8-sig')
summary = pd.read_csv(OUTPUT_PATH / 'subcategory_access_usage_correlation_summary_2025.csv', encoding='utf-8-sig')
unit_meta = pd.read_csv(OUTPUT_PATH / 'subcategory_access_usage_unit_meta_2025.csv', encoding='utf-8-sig')
wide = pd.read_csv(OUTPUT_PATH / 'subcategory_access_usage_gu_wide_2025.csv', encoding='utf-8-sig')

print('구-소분류 wide 구조:', wide.shape)
print('분석 소분류 수:', wide['소분류'].nunique())
print('분석그룹별 소분류 수')
display(wide.groupby('분석그룹')['소분류'].nunique().reset_index(name='소분류수'))

print('\n소분류별 시설수와 이용건수')
display(unit_meta.sort_values(['분석그룹', '중분류', '소분류']))

print('\nSpearman 상관계수')
display(spearman_table.round(4))

print('\nPearson 상관계수')
display(pearson_table.round(4))

print('\n분석그룹별 요약')
display(summary.round(4))

In [ ]:
display(Image(filename=str(IMAGE_PATH / 'subcategory_access_usage_spearman_2025.png')))
display(Image(filename=str(IMAGE_PATH / 'subcategory_access_usage_pearson_2025.png')))
display(Image(filename=str(IMAGE_PATH / 'subcategory_access_usage_group_spearman_2025.png')))
display(Image(filename=str(IMAGE_PATH / 'subcategory_access_usage_group_pearson_2025.png')))

## 11. 정부 최근접 접근성과 SFCA/H3SFCA 비교 split 그래프
- `subcategory_access_usage_pearson_table_2025.csv`, `subcategory_access_usage_spearman_table_2025.csv`를 읽어 재현함.
- 관심 지표가 정부 최근접 접근성보다 높은 소분류와 정부 최근접 접근성이 더 높은 소분류를 좌우로 분리함.
- 디자인 규칙: `Noto Sans KR`, 비교 지표 회색, 관심 지표 주황, grid 없음, 수치 표기.
- 이 그래프는 방법론 간 비교이므로 주황/회색을 사용함. 양수·음수 방향 자체가 핵심인 별도 그래프에서는 주황/청록 대비를 사용함.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.patches import Patch
import numpy as np
import pandas as pd


PROJECT_PATH = Path(r"C:\project\oracle_mnc_project")
EDA_PATH = PROJECT_PATH / "notebooks" / "eda"
OUTPUT_PATH = EDA_PATH / "OUTPUT"
IMAGE_PATH = EDA_PATH / "IMAGE"
IMAGE_PATH.mkdir(parents=True, exist_ok=True)


STYLE = {
    "background": "#fbf6ef",
    "text": "#241f1c",
    "axis": "#d9d3cb",
    "tick": "#766f67",
    "comparison": "#c7c2bc",
    "orange": "#f26b30",
    "teal": "#009688",
}

COMPARE_COL = "정부 최근접(-거리)"
FOCUS_COLUMNS = {
    "h3sfca": {
        "column": "선호 반영 H3SFCA",
        "legend": "선호 반영 H3SFCA",
        "left_title": "H3SFCA가 더 높은 소분류",
    },
    "sfca": {
        "column": "선호 미반영 SFCA",
        "legend": "선호 미반영 SFCA",
        "left_title": "SFCA가 더 높은 소분류",
    },
}


def configure_project_font():
    """Use the project font first, with OS fallbacks for reproducibility."""

    installed_fonts = {font.name for font in font_manager.fontManager.ttflist}
    for font_name in ["Noto Sans KR", "Malgun Gothic", "Apple SD Gothic Neo", "AppleGothic", "DejaVu Sans"]:
        if font_name in installed_fonts:
            plt.rcParams["font.family"] = font_name
            break
    else:
        plt.rcParams["font.family"] = "DejaVu Sans"
    plt.rcParams["axes.unicode_minus"] = False


def read_correlation_table(method):
    """Load the wide subcategory correlation table for Pearson or Spearman."""

    method = method.lower()
    if method not in {"pearson", "spearman"}:
        raise ValueError("method는 'pearson' 또는 'spearman'만 지원합니다.")
    path = OUTPUT_PATH / f"subcategory_access_usage_{method}_table_2025.csv"
    return pd.read_csv(path, encoding="utf-8-sig")


def split_by_stronger_indicator(table, focus_col):
    """Split rows by whether the focus indicator exceeds government nearest access."""

    required = ["소분류", focus_col, COMPARE_COL]
    missing = [column for column in required if column not in table.columns]
    if missing:
        raise ValueError(f"필수 열이 없습니다: {missing}")

    data = table[required].replace([np.inf, -np.inf], np.nan).dropna().copy()
    data[focus_col] = pd.to_numeric(data[focus_col], errors="coerce")
    data[COMPARE_COL] = pd.to_numeric(data[COMPARE_COL], errors="coerce")
    data = data.dropna(subset=[focus_col, COMPARE_COL])

    focus_higher = data.loc[data[focus_col] >= data[COMPARE_COL]].sort_values(
        focus_col,
        ascending=False,
    )
    government_higher = data.loc[data[focus_col] < data[COMPARE_COL]].sort_values(
        COMPARE_COL,
        ascending=False,
    )
    return focus_higher, government_higher


def label_bar(axis, value, y_position):
    """Place a two-decimal value label just outside the bar end."""

    offset = 0.02
    if value >= 0:
        axis.text(
            value + offset,
            y_position,
            f"{value:.2f}",
            va="center",
            ha="left",
            fontsize=15,
            color=STYLE["text"],
        )
    else:
        axis.text(
            value - offset,
            y_position,
            f"{value:.2f}",
            va="center",
            ha="right",
            fontsize=15,
            color=STYLE["text"],
        )


def draw_split_axis(axis, data, focus_col, subtitle, xlim):
    """Draw one side of the split horizontal comparison chart."""

    y_positions = np.arange(len(data))
    bar_height = 0.36

    axis.barh(
        y_positions - bar_height / 2,
        data[COMPARE_COL],
        height=bar_height,
        color=STYLE["comparison"],
        edgecolor="none",
    )
    axis.barh(
        y_positions + bar_height / 2,
        data[focus_col],
        height=bar_height,
        color=STYLE["orange"],
        edgecolor="none",
    )

    for y_position, (_, row) in zip(y_positions, data.iterrows()):
        label_bar(axis, row[COMPARE_COL], y_position - bar_height / 2)
        label_bar(axis, row[focus_col], y_position + bar_height / 2)

    axis.axvline(0, color=STYLE["axis"], linewidth=1.35, zorder=0)
    axis.set_xlim(xlim)
    axis.set_yticks(y_positions)
    axis.set_yticklabels(data["소분류"], fontsize=18, color=STYLE["text"])
    axis.invert_yaxis()
    axis.set_title(subtitle, fontsize=23, fontweight=900, color=STYLE["text"], pad=24)
    axis.grid(False)
    axis.tick_params(axis="x", labelsize=13, colors=STYLE["tick"])
    axis.tick_params(axis="y", length=0)

    for spine in ["top", "right", "left"]:
        axis.spines[spine].set_visible(False)
    axis.spines["bottom"].set_color(STYLE["axis"])
    axis.set_facecolor(STYLE["background"])


def calculate_shared_xlim(*frames, columns):
    """Use one x-axis range for both panels so the bar lengths are comparable."""

    values = []
    for frame in frames:
        for column in columns:
            values.extend(frame[column].dropna().astype(float).tolist())
    if not values:
        return (-0.45, 0.8)
    lower = min(-0.45, np.floor((min(values) - 0.05) * 10) / 10)
    upper = max(0.8, np.ceil((max(values) + 0.05) * 10) / 10)
    return (lower, upper)


def plot_government_vs_focus_split(method, focus_key):
    """Create the split chart comparing government nearest access to SFCA/H3SFCA."""

    configure_project_font()
    focus = FOCUS_COLUMNS[focus_key]
    focus_col = focus["column"]
    table = read_correlation_table(method)
    focus_higher, government_higher = split_by_stronger_indicator(table, focus_col)
    xlim = calculate_shared_xlim(
        focus_higher,
        government_higher,
        columns=[focus_col, COMPARE_COL],
    )

    metric_title = "Pearson" if method.lower() == "pearson" else "Spearman"
    fig, axes = plt.subplots(1, 2, figsize=(22, 10.5), dpi=180)
    fig.patch.set_facecolor(STYLE["background"])

    draw_split_axis(
        axes[0],
        focus_higher,
        focus_col,
        focus["left_title"],
        xlim,
    )
    draw_split_axis(
        axes[1],
        government_higher,
        focus_col,
        "최근접 접근성이 더 높은 소분류",
        xlim,
    )

    legend_handles = [
        Patch(facecolor=STYLE["orange"], edgecolor="none", label=focus["legend"]),
        Patch(facecolor=STYLE["comparison"], edgecolor="none", label=COMPARE_COL),
    ]
    fig.suptitle(
        f"소분류별 {metric_title} 상관계수 비교",
        fontsize=35,
        fontweight=900,
        color=STYLE["text"],
        y=0.985,
    )
    fig.legend(
        handles=legend_handles,
        loc="upper center",
        bbox_to_anchor=(0.5, 0.925),
        ncol=2,
        frameon=False,
        fontsize=16,
        handlelength=1.8,
        columnspacing=2.6,
    )
    fig.tight_layout(rect=(0, 0, 1, 0.875), w_pad=5.0)

    output_path = IMAGE_PATH / f"subcategory_{method.lower()}_government_vs_{focus_key}_split_2025.png"
    fig.savefig(output_path, dpi=180, bbox_inches="tight", facecolor=STYLE["background"])
    plt.close(fig)
    return output_path


generated_split_images = []
for method_name in ["pearson", "spearman"]:
    for focus_key in ["h3sfca", "sfca"]:
        generated_split_images.append(plot_government_vs_focus_split(method_name, focus_key))

print("생성된 split 그래프")
for path in generated_split_images:
    print("-", path)

In [ ]:
display(Image(filename=str(IMAGE_PATH / 'subcategory_pearson_government_vs_h3sfca_split_2025.png')))
display(Image(filename=str(IMAGE_PATH / 'subcategory_spearman_government_vs_h3sfca_split_2025.png')))
display(Image(filename=str(IMAGE_PATH / 'subcategory_pearson_government_vs_sfca_split_2025.png')))
display(Image(filename=str(IMAGE_PATH / 'subcategory_spearman_government_vs_sfca_split_2025.png')))

## 12. 상관계수 유의성 검정
- 상관계수가 0과 다른지는 Pearson/Spearman 상관검정으로 확인함.
- 두 접근성 지표의 상관계수 차이는 같은 종속변수(`대상자1인당_이용건수`)를 공유하므로 Williams test로 확인함.
- 비교쌍은 `선호 반영 H3SFCA vs 정부 최근접(-거리)`, `선호 미반영 SFCA vs 정부 최근접(-거리)`임.
- p-value는 다중검정 보정 전 탐색값이며, `p < 0.05`를 유의 기준으로 표시함.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from scipy import stats


EDA_PATH = Path(r"C:\project\oracle_mnc_project\notebooks\eda")
OUTPUT_PATH = EDA_PATH / "OUTPUT"

TARGET_COL = "대상자1인당_이용건수"
GOV_COL = "정부최근접접근성_음거리"
INDICATOR_COLUMNS = {
    "선호 반영 H3SFCA": "H3SFCA_선호반영",
    "선호 미반영 SFCA": "SFCA_선호미반영",
    "정부 최근접(-거리)": GOV_COL,
}
COMPARISON_PAIRS = {
    "H3SFCA - 정부최근접": "H3SFCA_선호반영",
    "SFCA - 정부최근접": "SFCA_선호미반영",
}


def clean_pair(frame, *columns):
    data = frame[list(columns)].replace([np.inf, -np.inf], np.nan).dropna().copy()
    for column in columns:
        data[column] = pd.to_numeric(data[column], errors="coerce")
    return data.dropna()


def correlation_test(x, y, method):
    data = pd.concat([x, y], axis=1).replace([np.inf, -np.inf], np.nan).dropna()
    if len(data) < 4 or data.iloc[:, 0].nunique() < 2 or data.iloc[:, 1].nunique() < 2:
        return np.nan, np.nan, len(data)
    if method == "Pearson":
        result = stats.pearsonr(data.iloc[:, 0], data.iloc[:, 1])
    elif method == "Spearman":
        result = stats.spearmanr(data.iloc[:, 0], data.iloc[:, 1])
    else:
        raise ValueError(f"지원하지 않는 method입니다: {method}")
    return float(result.statistic), float(result.pvalue), len(data)


def williams_test(r_y_focus, r_y_gov, r_focus_gov, n):
    """Compare two dependent correlations sharing y by Williams test."""

    if n <= 3 or any(pd.isna(v) for v in [r_y_focus, r_y_gov, r_focus_gov]):
        return np.nan, np.nan, n - 3

    determinant = (
        1
        - r_y_focus**2
        - r_y_gov**2
        - r_focus_gov**2
        + 2 * r_y_focus * r_y_gov * r_focus_gov
    )
    denominator = (
        2 * ((n - 1) / (n - 3)) * determinant
        + ((r_y_focus + r_y_gov) ** 2 / 4) * (1 - r_focus_gov) ** 3
    )
    if denominator <= 0 or not np.isfinite(denominator):
        return np.nan, np.nan, n - 3

    t_stat = (r_y_focus - r_y_gov) * np.sqrt((n - 1) * (1 + r_focus_gov) / denominator)
    p_value = 2 * stats.t.sf(abs(t_stat), df=n - 3)
    return float(t_stat), float(p_value), n - 3


def dependent_correlation_difference_test(group, focus_col, method):
    data = clean_pair(group, TARGET_COL, focus_col, GOV_COL)
    if method == "Spearman":
        data = data.rank(method="average")

    if len(data) < 4:
        return {
            "n": len(data),
            "관심지표_r": np.nan,
            "정부최근접_r": np.nan,
            "관심-정부_차이": np.nan,
            "관심_정부간_r": np.nan,
            "t": np.nan,
            "df": len(data) - 3,
            "p_value": np.nan,
        }

    r_y_focus = data[TARGET_COL].corr(data[focus_col], method="pearson")
    r_y_gov = data[TARGET_COL].corr(data[GOV_COL], method="pearson")
    r_focus_gov = data[focus_col].corr(data[GOV_COL], method="pearson")
    t_stat, p_value, df = williams_test(r_y_focus, r_y_gov, r_focus_gov, len(data))

    return {
        "n": len(data),
        "관심지표_r": r_y_focus,
        "정부최근접_r": r_y_gov,
        "관심-정부_차이": r_y_focus - r_y_gov,
        "관심_정부간_r": r_focus_gov,
        "t": t_stat,
        "df": df,
        "p_value": p_value,
    }


wide = pd.read_csv(OUTPUT_PATH / "subcategory_access_usage_gu_wide_2025.csv", encoding="utf-8-sig")

zero_rows = []
for (analysis_group, mid, sub), group in wide.groupby(["분석그룹", "중분류", "소분류"]):
    for indicator_name, indicator_col in INDICATOR_COLUMNS.items():
        for method in ["Pearson", "Spearman"]:
            coefficient, p_value, n = correlation_test(group[indicator_col], group[TARGET_COL], method)
            zero_rows.append({
                "검정": "상관계수=0",
                "방법": method,
                "분석그룹": analysis_group,
                "중분류": mid,
                "소분류": sub,
                "지표": indicator_name,
                "n": n,
                "r": coefficient,
                "p_value": p_value,
                "유의_0.05": p_value < 0.05 if pd.notna(p_value) else False,
            })
zero_test = pd.DataFrame(zero_rows)

difference_rows = []
for (analysis_group, mid, sub), group in wide.groupby(["분석그룹", "중분류", "소분류"]):
    for comparison_name, focus_col in COMPARISON_PAIRS.items():
        for method in ["Pearson", "Spearman"]:
            result = dependent_correlation_difference_test(group, focus_col, method)
            difference_rows.append({
                "검정": "상관계수 차이=0",
                "방법": method,
                "분석그룹": analysis_group,
                "중분류": mid,
                "소분류": sub,
                "비교": comparison_name,
                **result,
                "유의_0.05": result["p_value"] < 0.05 if pd.notna(result["p_value"]) else False,
                "방향": (
                    "관심지표 우위"
                    if pd.notna(result["관심-정부_차이"]) and result["관심-정부_차이"] > 0
                    else "정부최근접 우위"
                ),
            })
difference_test = pd.DataFrame(difference_rows)

zero_summary = (
    zero_test
    .groupby(["방법", "지표"], as_index=False)
    .agg(
        검정수=("소분류", "count"),
        유의_소분류수=("유의_0.05", "sum"),
        평균_r=("r", "mean"),
        절대r_평균=("r", lambda x: x.abs().mean()),
        최소_p=("p_value", "min"),
    )
)
difference_summary = (
    difference_test
    .groupby(["방법", "비교"], as_index=False)
    .agg(
        검정수=("소분류", "count"),
        관심지표_우위수=("방향", lambda x: (x == "관심지표 우위").sum()),
        정부최근접_우위수=("방향", lambda x: (x == "정부최근접 우위").sum()),
        유의_차이수=("유의_0.05", "sum"),
        평균_차이=("관심-정부_차이", "mean"),
        최소_p=("p_value", "min"),
    )
)

print("상관계수 0 검정 요약")
display(zero_summary.round(4))

print("\np < 0.05인 상관계수 0 검정 결과")
display(
    zero_test[zero_test["유의_0.05"]]
    .sort_values(["방법", "p_value"])
    [["방법", "분석그룹", "중분류", "소분류", "지표", "n", "r", "p_value"]]
    .round(4)
)

print("\n상관계수 차이 검정 요약: Williams test")
display(difference_summary.round(4))

print("\np < 0.05인 상관계수 차이 검정 결과")
display(
    difference_test[difference_test["유의_0.05"]]
    .sort_values(["방법", "p_value"])
    [["방법", "분석그룹", "중분류", "소분류", "비교", "n", "관심지표_r", "정부최근접_r", "관심-정부_차이", "t", "df", "p_value", "방향"]]
    .round(4)
)

print("\nPearson 기준 첨부 그래프 대응 상세: H3SFCA vs 정부 최근접")
display(
    difference_test[
        difference_test["방법"].eq("Pearson")
        & difference_test["비교"].eq("H3SFCA - 정부최근접")
    ]
    .sort_values("관심-정부_차이", ascending=False)
    [["분석그룹", "중분류", "소분류", "n", "관심지표_r", "정부최근접_r", "관심-정부_차이", "t", "df", "p_value", "유의_0.05", "방향"]]
    .round(4)
)

## 13. 25km 기준 소분류 접근성 재계산
- 기존 산출물은 덮어쓰지 않고 _25km_recalc suffix로 별도 저장한다.
- 현재 
etwork_competition_25km upstream 가맹점을 기준으로 소분류 시설수, 정부 최근접 접근성, SFCA/H3SFCA, 상관계수 표, split 그래프를 다시 계산한다.
- 외부 25km 가맹점 공급량은 H3SFCA 노트북과 동일하게 서울 동일 소분류/중분류 공급량 중앙값으로 보정한다.


In [ ]:
from __future__ import annotations

import gc
import heapq
import time
import warnings
from collections import defaultdict
from pathlib import Path

import geopandas as gpd
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from matplotlib import font_manager
from matplotlib.patches import Patch
from scipy.spatial import cKDTree


warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.max_columns", 120)

PROJECT_PATH = Path(r"C:\project\oracle_mnc_project")
EDA_PATH = PROJECT_PATH / "notebooks" / "eda"
OUTPUT_PATH = EDA_PATH / "OUTPUT"
IMAGE_PATH = EDA_PATH / "IMAGE"
ACCESS_OUTPUT_PATH = PROJECT_PATH / "notebooks" / "access" / "OUTPUT"
H3_PATH = ACCESS_OUTPUT_PATH / "h3sfca"
NETWORK_OUTPUT_PATH = PROJECT_PATH / "analysis_table" / "data" / "output" / "network_competition_25km"
DATA_INPUT_PATH = PROJECT_PATH / "analysis_table" / "data" / "input"
KTDB_PATH = DATA_INPUT_PATH / "network" / "ktdb_transport_network"
MNC_CARD_PATH = PROJECT_PATH / "data" / "raw" / "mnc_card" / "mnc_seoul_usage_issuance_2021_2025.xlsx"

OUTPUT_PATH.mkdir(parents=True, exist_ok=True)
IMAGE_PATH.mkdir(parents=True, exist_ok=True)

GRID_COMPETITION_PATH = NETWORK_OUTPUT_PATH / "경쟁권25km_분석격자.parquet"
STORE_COMPETITION_PATH = NETWORK_OUTPUT_PATH / "경쟁권25km_분석가맹점.parquet"
WALK_PAIR_PATH = NETWORK_OUTPUT_PATH / "경쟁권25km_도보접근성.parquet"
TRANSIT_PAIR_PATH = NETWORK_OUTPUT_PATH / "경쟁권25km_대중교통접근성.parquet"
ANALYSIS_AREA_PATH = NETWORK_OUTPUT_PATH / "경쟁권25km_분석권역.gpkg"
SUPPLY_PATH = H3_PATH / "문화누리_가맹점_카테고리별_공급량.csv"
H3_DEMAND_PATH = H3_PATH / "h3sfca_격자_중분류_선호수요.parquet"
SEOUL_GRID_POP_PATH = H3_PATH / "sfca_no_preference_격자_중분류_접근성.csv"
OLD_UNIT_META_PATH = OUTPUT_PATH / "subcategory_access_usage_unit_meta_2025.csv"

NODE_PATH = next(KTDB_PATH.rglob("2025node.txt"))
LINK_PATH = next(KTDB_PATH.rglob("2025link.txt"))

SERVICE_LIMIT_M = 10_000
ROAD_BUFFER_M = 2_000
ROAD_LINK_TYPES = list(range(101, 109))
PRIMARY_LAMBDA = 1.2

WALK_CATEGORIES = ["도서", "문화체험", "음악", "영상", "체육시설", "체육용품"]
TRANSIT_CATEGORIES = ["미술", "공연", "스포츠관람", "관광지"]
ML_CATEGORIES = ["도서", "공연", "미술", "문화체험", "관광지", "스포츠관람", "체육시설", "영상"]
NON_ML_CATEGORIES = ["음악", "체육용품"]
CATEGORY_MODE = {
    **{category: "도보" for category in WALK_CATEGORIES},
    **{category: "대중교통" for category in TRANSIT_CATEGORIES},
}

STYLE = {
    "background": "#fbf6ef",
    "text": "#241f1c",
    "axis": "#d9d3cb",
    "tick": "#766f67",
    "comparison": "#c7c2bc",
    "orange": "#f26b30",
    "teal": "#009688",
}


def clean_number(series: pd.Series) -> pd.Series:
    return pd.to_numeric(
        series.astype(str).str.replace(",", "", regex=False).str.strip(),
        errors="coerce",
    )


def normalize_label(value: object) -> str:
    return (
        str(value)
        .replace("\n", "")
        .replace(" ", "")
        .replace("ㆍ", "·")
        .strip()
    )


def find_usage_column(columns, subcategory: str) -> str:
    target = normalize_label(f"{subcategory}(건)")
    for column in columns:
        if normalize_label(column) == target:
            return column
    raise KeyError(f"이용건수 컬럼을 찾지 못함: {subcategory}")


def apply_decay(cost: pd.Series, mode: pd.Series) -> np.ndarray:
    cost = pd.to_numeric(cost, errors="coerce")
    mode = pd.Series(mode, index=cost.index)
    weight = np.zeros(len(cost), dtype="float32")

    walk_idx = mode.eq("도보")
    transit_idx = mode.eq("대중교통")

    weight[walk_idx & (cost >= 0) & (cost <= 250)] = 1.0
    weight[walk_idx & (cost > 250) & (cost <= 500)] = 0.6
    weight[walk_idx & (cost > 500) & (cost <= 750)] = 0.25
    weight[transit_idx & (cost >= 0) & (cost <= 7)] = 1.0
    weight[transit_idx & (cost > 7) & (cost <= 14)] = 0.6
    weight[transit_idx & (cost > 14) & (cost <= 20)] = 0.25

    return weight


def weighted_mean(value: pd.Series, weight: pd.Series) -> float:
    value = pd.to_numeric(value, errors="coerce")
    weight = pd.to_numeric(weight, errors="coerce").fillna(0)
    valid = value.notna()
    if valid.sum() == 0:
        return np.nan
    value = value[valid]
    weight = weight[valid]
    if weight.sum() > 0:
        return float(np.average(value, weights=weight))
    return float(value.mean())


def configure_project_font() -> None:
    installed_fonts = {font.name for font in font_manager.fontManager.ttflist}
    for font_name in ["Noto Sans KR", "Malgun Gothic", "Apple SD Gothic Neo", "AppleGothic", "DejaVu Sans"]:
        if font_name in installed_fonts:
            plt.rcParams["font.family"] = font_name
            break
    else:
        plt.rcParams["font.family"] = "DejaVu Sans"
    plt.rcParams["axes.unicode_minus"] = False


def load_selected_subcategories() -> pd.DataFrame:
    selected = pd.read_csv(OLD_UNIT_META_PATH, encoding="utf-8-sig")[
        ["분석그룹", "중분류", "소분류"]
    ].drop_duplicates().copy()
    selected["접근수단"] = selected["중분류"].map(CATEGORY_MODE)
    missing = selected[selected["접근수단"].isna()]
    if not missing.empty:
        raise ValueError(f"접근수단 매핑 누락: {missing[['중분류', '소분류']].to_dict('records')}")
    return selected


def load_usage(selected: pd.DataFrame, seoul_grid_base: pd.DataFrame) -> pd.DataFrame:
    gu_mnc_pop = (
        seoul_grid_base
        .groupby("시군구", as_index=False)
        .agg(
            구별_추정인구=("추정_인구수", "sum"),
            구별_문화누리대상자추정인구=("문화누리대상자_추정_인구수", "sum"),
            격자수=("GRID_CD", "nunique"),
        )
    )

    usage_raw = pd.read_excel(MNC_CARD_PATH, sheet_name="2025")
    usage_raw["광역"] = usage_raw["광역"].astype(str).str.strip()
    usage_raw = usage_raw[usage_raw["광역"].eq("서울")].copy()
    usage_raw = usage_raw.rename(columns={"기초": "시군구"})
    usage_raw["시군구"] = usage_raw["시군구"].astype(str).str.strip()

    usage_list = []
    for _, row in selected.iterrows():
        col = find_usage_column(usage_raw.columns, row["소분류"])
        temp = usage_raw[["시군구", col]].copy()
        temp = temp.rename(columns={col: "이용건수"})
        temp["분석그룹"] = row["분석그룹"]
        temp["중분류"] = row["중분류"]
        temp["소분류"] = row["소분류"]
        usage_list.append(temp)

    usage = pd.concat(usage_list, ignore_index=True)
    usage["이용건수"] = clean_number(usage["이용건수"]).fillna(0)
    usage = usage.merge(gu_mnc_pop, on="시군구", how="left")
    usage["대상자1인당_이용건수"] = np.where(
        usage["구별_문화누리대상자추정인구"] > 0,
        usage["이용건수"] / usage["구별_문화누리대상자추정인구"],
        np.nan,
    )
    usage["대상자천명당_이용건수"] = usage["대상자1인당_이용건수"] * 1000
    return usage


def load_store_25km(selected: pd.DataFrame) -> pd.DataFrame:
    store = gpd.read_parquet(STORE_COMPETITION_PATH).drop(columns="geometry", errors="ignore")
    store = store.merge(selected, on=["중분류", "소분류"], how="inner")

    supply = pd.read_csv(SUPPLY_PATH, encoding="utf-8-sig")
    supply = supply[supply["분석포함"].astype(bool)].copy()
    supply["공급량"] = pd.to_numeric(supply["공급량"], errors="coerce").fillna(1).clip(lower=0)

    store = store.merge(
        supply[["가맹점_ID", "중분류", "소분류", "공급량", "공급량_raw", "공급기준"]],
        on=["가맹점_ID", "중분류", "소분류"],
        how="left",
    )

    sub_ref = (
        supply.groupby(["중분류", "소분류"], as_index=False)["공급량"]
        .median()
        .rename(columns={"공급량": "소분류_보정공급량"})
    )
    mid_ref = (
        supply.groupby("중분류", as_index=False)["공급량"]
        .median()
        .rename(columns={"공급량": "중분류_보정공급량"})
    )
    store = store.merge(sub_ref, on=["중분류", "소분류"], how="left")
    store = store.merge(mid_ref, on="중분류", how="left")
    store["공급량"] = (
        pd.to_numeric(store["공급량"], errors="coerce")
        .fillna(store["소분류_보정공급량"])
        .fillna(store["중분류_보정공급량"])
        .fillna(1)
        .clip(lower=0)
    )
    store["공급기준"] = store["공급기준"].fillna("서울 동일 소분류/중분류 공급량 중앙값 대체")
    return store.drop(columns=["소분류_보정공급량", "중분류_보정공급량"], errors="ignore")


def build_road_graph():
    started = time.time()
    print("[정부최근접] KTDB node/link 로딩", flush=True)
    node = pd.read_csv(
        NODE_PATH,
        sep=r"\s+",
        skiprows=1,
        header=None,
        names=["record_type", "node_id", "x", "y"],
        engine="python",
    ).drop(columns="record_type")

    link = pd.read_csv(
        LINK_PATH,
        sep=r"\s+",
        skiprows=1,
        header=None,
        names=[
            "record_type", "from_node", "to_node", "length_km", "mode_code",
            "link_type", "lane", "capacity", "speed", "vdf", "cost",
        ],
        engine="python",
    ).drop(columns="record_type")

    node["node_id"] = pd.to_numeric(node["node_id"], errors="coerce")
    node["x"] = pd.to_numeric(node["x"], errors="coerce")
    node["y"] = pd.to_numeric(node["y"], errors="coerce")
    node = node.dropna(subset=["node_id", "x", "y"]).copy()
    node["node_id"] = node["node_id"].astype("int64")

    for col in ["from_node", "to_node", "length_km", "link_type"]:
        link[col] = pd.to_numeric(link[col], errors="coerce")
    link = link.dropna(subset=["from_node", "to_node", "length_km", "link_type"]).copy()
    link["from_node"] = link["from_node"].astype("int64")
    link["to_node"] = link["to_node"].astype("int64")
    link["link_type"] = link["link_type"].astype("int64")
    link["length_m"] = link["length_km"] * 1000

    ktdb_crs = (
        "+proj=tmerc +lat_0=38 +lon_0=128 +k=0.9999 "
        "+x_0=400000 +y_0=600000 +ellps=bessel "
        "+towgs84=-146.43,507.89,681.46 +units=m +no_defs"
    )

    node_gdf = gpd.GeoDataFrame(
        node.copy(),
        geometry=gpd.points_from_xy(node["x"], node["y"]),
        crs=ktdb_crs,
    ).to_crs("EPSG:5179")

    print("[정부최근접] 25km 권역 도로 노드 추출", flush=True)
    analysis_area = gpd.read_file(ANALYSIS_AREA_PATH).to_crs("EPSG:5179").dissolve()[["geometry"]].reset_index(drop=True)
    analysis_area_buffer = analysis_area.copy()
    analysis_area_buffer["geometry"] = analysis_area_buffer.geometry.buffer(ROAD_BUFFER_M)

    node_in_buffer = gpd.sjoin(
        node_gdf,
        analysis_area_buffer[["geometry"]],
        how="inner",
        predicate="within",
    ).drop(columns="index_right", errors="ignore")

    link_road = link[
        (link["length_m"] > 0)
        & (link["link_type"].isin(ROAD_LINK_TYPES))
    ].copy()

    buffer_node_ids = set(node_in_buffer["node_id"])
    link_road_area = link_road[
        link_road["from_node"].isin(buffer_node_ids)
        | link_road["to_node"].isin(buffer_node_ids)
    ].copy()

    road_node_ids = set(link_road_area["from_node"]) | set(link_road_area["to_node"])
    node_road_area = node_gdf[node_gdf["node_id"].isin(road_node_ids)].copy()

    edge = (
        link_road_area[["from_node", "to_node", "length_m"]]
        .groupby(["from_node", "to_node"], as_index=False)["length_m"]
        .min()
    )

    print("[정부최근접] 도로 그래프 구성", flush=True)
    graph = defaultdict(list)
    for from_node, to_node, length_m in edge[["from_node", "to_node", "length_m"]].itertuples(index=False, name=None):
        graph[int(from_node)].append((int(to_node), float(length_m)))
        graph[int(to_node)].append((int(from_node), float(length_m)))
    print(f"[정부최근접] 도로 그래프 완료: {len(graph):,}개 노드, {len(edge):,}개 링크, {time.time() - started:.1f}초", flush=True)
    return graph, node_road_area


def nearest_facility_dijkstra(graph, source_cost, cutoff):
    dist = {}
    nearest_node = {}
    heap = []

    for node_id, cost in source_cost.items():
        if pd.isna(cost) or node_id not in graph:
            continue
        cost = float(cost)
        if cost < dist.get(node_id, np.inf):
            dist[node_id] = cost
            nearest_node[node_id] = node_id
            heapq.heappush(heap, (cost, node_id, node_id))

    while heap:
        cost, node_id, source_node = heapq.heappop(heap)
        if cost != dist.get(node_id):
            continue
        if cost > cutoff:
            continue
        for next_node, edge_weight in graph[node_id]:
            next_cost = cost + edge_weight
            if next_cost < dist.get(next_node, np.inf) and next_cost <= cutoff:
                dist[next_node] = next_cost
                nearest_node[next_node] = source_node
                heapq.heappush(heap, (next_cost, next_node, source_node))

    return dist, nearest_node


def calculate_public_nearest(selected: pd.DataFrame, store_25km: pd.DataFrame, seoul_grid_base: pd.DataFrame) -> pd.DataFrame:
    road_graph, road_node = build_road_graph()
    print("[정부최근접] 격자/시설 최근접 도로 노드 매칭", flush=True)
    node_xy = np.column_stack([road_node.geometry.x.to_numpy(), road_node.geometry.y.to_numpy()])
    node_id_array = road_node["node_id"].to_numpy()
    node_tree = cKDTree(node_xy)

    public_grid = gpd.read_parquet(GRID_COMPETITION_PATH).to_crs("EPSG:5179")
    public_grid = public_grid[public_grid["서울여부"] == True].copy()
    public_grid = public_grid[["GRID_CD", "시군구", "행정동", "추정_인구수", "geometry"]].copy()
    public_grid = public_grid.merge(
        seoul_grid_base[["GRID_CD", "문화누리대상자_추정_인구수"]],
        on="GRID_CD",
        how="left",
    )
    public_grid["문화누리대상자_추정_인구수"] = pd.to_numeric(
        public_grid["문화누리대상자_추정_인구수"],
        errors="coerce",
    ).fillna(0)

    store_gdf = gpd.read_parquet(STORE_COMPETITION_PATH).to_crs("EPSG:5179")
    public_store = store_gdf.merge(
        selected[["분석그룹", "중분류", "소분류"]],
        on=["중분류", "소분류"],
        how="inner",
    ).dropna(subset=["geometry"]).copy()

    grid_xy = np.column_stack([public_grid.geometry.centroid.x.to_numpy(), public_grid.geometry.centroid.y.to_numpy()])
    grid_distance, grid_nearest_pos = node_tree.query(grid_xy, k=1)
    public_grid["도로망_노드ID"] = node_id_array[grid_nearest_pos]
    public_grid["격자_스냅거리_m"] = grid_distance

    store_xy = np.column_stack([public_store.geometry.x.to_numpy(), public_store.geometry.y.to_numpy()])
    store_distance, store_nearest_pos = node_tree.query(store_xy, k=1)
    public_store["도로망_노드ID"] = node_id_array[store_nearest_pos]
    public_store["시설_스냅거리_m"] = store_distance

    facility_counts = (
        store_25km
        .groupby(["분석그룹", "중분류", "소분류"], as_index=False)
        .agg(
            시설수=("가맹점_ID", "nunique"),
            시설수_서울=("서울여부", lambda x: int(pd.Series(x).astype(bool).sum())),
            시설수_외부25km=("서울여부", lambda x: int((~pd.Series(x).astype(bool)).sum())),
        )
    )

    public_result_list = []
    for _, sub_row in selected.iterrows():
        group = sub_row["분석그룹"]
        mid = sub_row["중분류"]
        sub = sub_row["소분류"]
        category_store = public_store[
            (public_store["분석그룹"] == group)
            & (public_store["중분류"] == mid)
            & (public_store["소분류"] == sub)
        ].copy()

        category_node_best = (
            category_store
            .sort_values(["도로망_노드ID", "시설_스냅거리_m", "가맹점_ID"])
            .drop_duplicates("도로망_노드ID")
            .set_index("도로망_노드ID")
        )
        source_cost = category_node_best["시설_스냅거리_m"].to_dict()
        node_cost, _nearest_node = nearest_facility_dijkstra(road_graph, source_cost, SERVICE_LIMIT_M)

        temp = public_grid[
            ["GRID_CD", "시군구", "행정동", "문화누리대상자_추정_인구수", "도로망_노드ID", "격자_스냅거리_m"]
        ].copy()
        temp["분석그룹"] = group
        temp["중분류"] = mid
        temp["소분류"] = sub
        temp["노드_시설접근비용_m"] = temp["도로망_노드ID"].map(node_cost)
        temp["문화시설_접근거리_m"] = temp["격자_스냅거리_m"] + temp["노드_시설접근비용_m"]
        temp["문화시설_10km_접근가능"] = temp["문화시설_접근거리_m"] <= SERVICE_LIMIT_M
        public_result_list.append(temp)
        print(f"[정부최근접] {sub}: 25km 시설 {len(category_store):,}개, 거리결측 {temp['문화시설_접근거리_m'].isna().sum():,}개", flush=True)

    public_grid_access = pd.concat(public_result_list, ignore_index=True)

    public_gu = (
        public_grid_access
        .groupby(["시군구", "분석그룹", "중분류", "소분류"], as_index=False)
        .apply(
            lambda x: pd.Series({
                "정부최근접_가중평균거리_m": weighted_mean(
                    x["문화시설_접근거리_m"],
                    x["문화누리대상자_추정_인구수"],
                ),
                "서비스권역_인구비중": (
                    x.loc[x["문화시설_10km_접근가능"], "문화누리대상자_추정_인구수"].sum()
                    / x["문화누리대상자_추정_인구수"].sum()
                ) if x["문화누리대상자_추정_인구수"].sum() > 0 else np.nan,
                "서비스권역_도달대상자수": x.loc[x["문화시설_10km_접근가능"], "문화누리대상자_추정_인구수"].sum(),
                "대상자수": x["문화누리대상자_추정_인구수"].sum(),
                "거리결측격자수": x["문화시설_접근거리_m"].isna().sum(),
            }),
            include_groups=False,
        )
    )
    public_gu = public_gu.merge(facility_counts, on=["분석그룹", "중분류", "소분류"], how="left")

    del road_graph, road_node, public_grid, public_store, public_grid_access
    gc.collect()
    return public_gu


def iter_selected_pairs(pair_path: Path, store_info: pd.DataFrame, mode_name: str, batch_size: int = 1_000_000):
    mode_store = store_info[store_info["접근수단"] == mode_name].copy()
    if mode_store.empty:
        return
    store_ids = set(mode_store["가맹점_ID"])
    parquet_file = pq.ParquetFile(pair_path)
    started = time.time()
    for batch_index, batch in enumerate(parquet_file.iter_batches(
        columns=["GRID_CD", "가맹점_ID", "접근수단", "접근비용"],
        batch_size=batch_size,
    ), start=1):
        pair = batch.to_pandas()
        pair = pair[pair["가맹점_ID"].isin(store_ids)].copy()
        if pair.empty:
            continue
        pair = pair.merge(
            mode_store[["가맹점_ID", "분석그룹", "중분류", "소분류", "소분류분석명", "접근수단", "공급량"]],
            on=["가맹점_ID", "접근수단"],
            how="inner",
        )
        pair["거리감쇠"] = apply_decay(pair["접근비용"], pair["접근수단"])
        pair = pair[pair["거리감쇠"] > 0].copy()
        if not pair.empty:
            if batch_index % 5 == 0:
                print(
                    f"[{mode_name} pair] {batch_index}/{parquet_file.num_row_groups} batch, "
                    f"{time.time() - started:.1f}초",
                    flush=True,
                )
            yield pair


def calculate_sfca_h3(store_25km: pd.DataFrame, selected: pd.DataFrame, grid_all: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    store_info = store_25km[
        ["가맹점_ID", "분석그룹", "중분류", "소분류", "접근수단", "공급량"]
    ].drop_duplicates().copy()
    store_info["소분류분석명"] = (
        store_info["분석그룹"].astype(str) + "|" +
        store_info["중분류"].astype(str) + "|" +
        store_info["소분류"].astype(str)
    )

    grid_demand = grid_all[["GRID_CD", "추정_인구수", "서울여부"]].copy()
    h3_demand = pd.read_parquet(H3_DEMAND_PATH)
    h3_demand = h3_demand[h3_demand["중분류"].isin(ML_CATEGORIES)].copy()
    h3_demand["선호수요"] = (
        pd.to_numeric(h3_demand["일반인구_선호수요"], errors="coerce").fillna(0)
        + PRIMARY_LAMBDA * pd.to_numeric(h3_demand["문화누리대상자_기본선호수요"], errors="coerce").fillna(0)
    )
    h3_demand = h3_demand.groupby(["GRID_CD", "중분류"], as_index=False)["선호수요"].sum()

    pair_sources = [("도보", WALK_PAIR_PATH), ("대중교통", TRANSIT_PAIR_PATH)]

    sfca_facility_demand_parts = []
    hden_parts = []
    pair_qc = []

    print("[SFCA/H3] 1차: 공급별 수요와 Huff 분모 집계", flush=True)
    for mode_name, pair_path in pair_sources:
        for pair in iter_selected_pairs(pair_path, store_info, mode_name):
            pair_qc.append({"단계": "1차", "접근수단": mode_name, "pair행수": len(pair)})
            pair = pair.merge(grid_demand[["GRID_CD", "추정_인구수"]], on="GRID_CD", how="left")
            pair["가중수요"] = pd.to_numeric(pair["추정_인구수"], errors="coerce").fillna(0) * pair["거리감쇠"]
            pair["매력도"] = pair["공급량"] * pair["거리감쇠"]
            sfca_facility_demand_parts.append(
                pair.groupby(["가맹점_ID", "소분류분석명"], as_index=False).agg(
                    공급량=("공급량", "first"),
                    가중수요=("가중수요", "sum"),
                )
            )
            hden_parts.append(
                pair[pair["중분류"].isin(ML_CATEGORIES)]
                .groupby(["GRID_CD", "소분류분석명"], as_index=False)["매력도"]
                .sum()
                .rename(columns={"매력도": "Huff분모"})
            )
            del pair
            gc.collect()

    print("[SFCA/H3] 1차 결과 병합", flush=True)
    sfca_facility = (
        pd.concat(sfca_facility_demand_parts, ignore_index=True)
        .groupby(["가맹점_ID", "소분류분석명"], as_index=False)
        .agg(공급량=("공급량", "first"), 가중수요=("가중수요", "sum"))
    )
    sfca_facility["공급수요비"] = np.where(
        sfca_facility["가중수요"] > 0,
        sfca_facility["공급량"] / sfca_facility["가중수요"],
        0,
    )

    hden = (
        pd.concat(hden_parts, ignore_index=True)
        .groupby(["GRID_CD", "소분류분석명"], as_index=False)["Huff분모"]
        .sum()
    )

    sfca_access_parts = []
    h3_facility_demand_parts = []

    print("[SFCA/H3] 2차: SFCA 격자 접근성과 H3 공급별 수요 집계", flush=True)
    for mode_name, pair_path in pair_sources:
        for pair in iter_selected_pairs(pair_path, store_info, mode_name):
            pair_qc.append({"단계": "2차", "접근수단": mode_name, "pair행수": len(pair)})
            pair = pair.merge(
                sfca_facility[["가맹점_ID", "소분류분석명", "공급수요비"]],
                on=["가맹점_ID", "소분류분석명"],
                how="left",
            )
            pair["접근성기여"] = pair["공급수요비"].fillna(0) * pair["거리감쇠"]
            pair = pair.merge(grid_demand[["GRID_CD", "서울여부"]], on="GRID_CD", how="left")
            pair_seoul = pair[pair["서울여부"] == True].copy()
            sfca_access_parts.append(
                pair_seoul.groupby(["GRID_CD", "분석그룹", "중분류", "소분류"], as_index=False).agg(
                    SFCA_선호미반영=("접근성기여", "sum"),
                    SFCA_접근가능가맹점수=("가맹점_ID", "nunique"),
                    SFCA_평균접근비용=("접근비용", "mean"),
                )
            )

            pair_h3 = pair[pair["중분류"].isin(ML_CATEGORIES)].copy()
            if not pair_h3.empty:
                pair_h3["매력도"] = pair_h3["공급량"] * pair_h3["거리감쇠"]
                pair_h3 = pair_h3.merge(hden, on=["GRID_CD", "소분류분석명"], how="left")
                pair_h3["Huff확률"] = np.where(pair_h3["Huff분모"] > 0, pair_h3["매력도"] / pair_h3["Huff분모"], 0)
                pair_h3 = pair_h3.merge(h3_demand, on=["GRID_CD", "중분류"], how="left")
                pair_h3["가중수요"] = pair_h3["선호수요"].fillna(0) * pair_h3["Huff확률"]
                h3_facility_demand_parts.append(
                    pair_h3.groupby(["가맹점_ID", "소분류분석명"], as_index=False).agg(
                        공급량=("공급량", "first"),
                        가중수요=("가중수요", "sum"),
                    )
                )
                del pair_h3

            del pair, pair_seoul
            gc.collect()

    print("[SFCA/H3] 2차 결과 병합", flush=True)
    sfca_grid = (
        pd.concat(sfca_access_parts, ignore_index=True)
        .groupby(["GRID_CD", "분석그룹", "중분류", "소분류"], as_index=False)
        .agg(
            SFCA_선호미반영=("SFCA_선호미반영", "sum"),
            SFCA_접근가능가맹점수=("SFCA_접근가능가맹점수", "sum"),
            SFCA_평균접근비용=("SFCA_평균접근비용", "mean"),
        )
    )

    h3_facility = (
        pd.concat(h3_facility_demand_parts, ignore_index=True)
        .groupby(["가맹점_ID", "소분류분석명"], as_index=False)
        .agg(공급량=("공급량", "first"), 가중수요=("가중수요", "sum"))
    )
    h3_facility["공급수요비"] = np.where(
        h3_facility["가중수요"] > 0,
        h3_facility["공급량"] / h3_facility["가중수요"],
        0,
    )

    h3_access_parts = []

    print("[SFCA/H3] 3차: H3SFCA 격자 접근성 집계", flush=True)
    for mode_name, pair_path in pair_sources:
        for pair in iter_selected_pairs(pair_path, store_info[store_info["중분류"].isin(ML_CATEGORIES)], mode_name):
            pair_qc.append({"단계": "3차", "접근수단": mode_name, "pair행수": len(pair)})
            pair["매력도"] = pair["공급량"] * pair["거리감쇠"]
            pair = pair.merge(hden, on=["GRID_CD", "소분류분석명"], how="left")
            pair["Huff확률"] = np.where(pair["Huff분모"] > 0, pair["매력도"] / pair["Huff분모"], 0)
            pair = pair.merge(
                h3_facility[["가맹점_ID", "소분류분석명", "공급수요비"]],
                on=["가맹점_ID", "소분류분석명"],
                how="left",
            )
            pair["접근성기여"] = pair["공급수요비"].fillna(0) * pair["Huff확률"]
            pair = pair.merge(h3_demand, on=["GRID_CD", "중분류"], how="left")
            pair = pair.merge(grid_demand[["GRID_CD", "서울여부"]], on="GRID_CD", how="left")
            pair = pair[pair["서울여부"] == True].copy()
            h3_access_parts.append(
                pair.groupby(["GRID_CD", "분석그룹", "중분류", "소분류"], as_index=False).agg(
                    H3SFCA_선호반영=("접근성기여", "sum"),
                    H3SFCA_접근가능가맹점수=("가맹점_ID", "nunique"),
                    H3SFCA_평균접근비용=("접근비용", "mean"),
                    선호수요=("선호수요", "first"),
                )
            )
            del pair
            gc.collect()

    print("[SFCA/H3] 3차 결과 병합", flush=True)
    h3_grid = (
        pd.concat(h3_access_parts, ignore_index=True)
        .groupby(["GRID_CD", "분석그룹", "중분류", "소분류"], as_index=False)
        .agg(
            H3SFCA_선호반영=("H3SFCA_선호반영", "sum"),
            H3SFCA_접근가능가맹점수=("H3SFCA_접근가능가맹점수", "sum"),
            H3SFCA_평균접근비용=("H3SFCA_평균접근비용", "mean"),
            선호수요=("선호수요", "first"),
        )
    )

    fallback_key = ["GRID_CD", "분석그룹", "중분류", "소분류"]
    h3_grid = sfca_grid[fallback_key + ["SFCA_선호미반영"]].merge(
        h3_grid,
        on=fallback_key,
        how="left",
    )
    h3_grid["H3SFCA_처리방식"] = np.where(
        h3_grid["중분류"].isin(NON_ML_CATEGORIES),
        "ML선호확률없음_SFCA대체",
        "선호반영계산",
    )
    h3_grid["H3SFCA_선호반영"] = np.where(
        h3_grid["중분류"].isin(NON_ML_CATEGORIES),
        h3_grid["SFCA_선호미반영"],
        h3_grid["H3SFCA_선호반영"].fillna(0),
    )
    h3_grid = h3_grid.drop(columns=["SFCA_선호미반영"])

    print("[SFCA/H3] pair QC")
    print(pd.DataFrame(pair_qc).groupby(["단계", "접근수단"], as_index=False)["pair행수"].sum())
    return sfca_grid, h3_grid


def aggregate_grid_access(seoul_grid_base: pd.DataFrame, selected: pd.DataFrame, access_df: pd.DataFrame, value_col: str, out_col: str) -> pd.DataFrame:
    full = seoul_grid_base[["GRID_CD", "시군구", "문화누리대상자_추정_인구수"]].merge(
        selected[["분석그룹", "중분류", "소분류"]],
        how="cross",
    )
    full = full.merge(
        access_df[["GRID_CD", "분석그룹", "중분류", "소분류", value_col]],
        on=["GRID_CD", "분석그룹", "중분류", "소분류"],
        how="left",
    )
    full[value_col] = pd.to_numeric(full[value_col], errors="coerce").fillna(0)
    return (
        full
        .groupby(["시군구", "분석그룹", "중분류", "소분류"], as_index=False)
        .apply(
            lambda x: pd.Series({
                out_col: weighted_mean(x[value_col], x["문화누리대상자_추정_인구수"]),
                "대상자수": x["문화누리대상자_추정_인구수"].sum(),
                "격자수": len(x),
            }),
            include_groups=False,
        )
    )


def safe_corr(df: pd.DataFrame, x_col: str, y_col: str, method: str) -> float:
    temp = df[[x_col, y_col]].replace([np.inf, -np.inf], np.nan).dropna()
    if len(temp) < 3:
        return np.nan
    if temp[x_col].nunique() < 2 or temp[y_col].nunique() < 2:
        return np.nan
    return float(temp[x_col].corr(temp[y_col], method=method))


def build_correlation_outputs(indicator_wide: pd.DataFrame, indicator_long: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    corr_list = []
    for (indicator_name, analysis_group, mid, sub), temp in indicator_long.groupby(["지표명", "분석그룹", "중분류", "소분류"]):
        corr_list.append({
            "지표명": indicator_name,
            "분석그룹": analysis_group,
            "중분류": mid,
            "소분류": sub,
            "n": temp[["지표값", "대상자1인당_이용건수"]].replace([np.inf, -np.inf], np.nan).dropna().shape[0],
            "Pearson": safe_corr(temp, "지표값", "대상자1인당_이용건수", "pearson"),
            "Spearman": safe_corr(temp, "지표값", "대상자1인당_이용건수", "spearman"),
            "지표값_평균": temp["지표값"].mean(),
            "대상자천명당_이용건수_평균": temp["대상자천명당_이용건수"].mean(),
        })
    corr = pd.DataFrame(corr_list)

    summary = (
        corr
        .groupby(["분석그룹", "지표명"], as_index=False)
        .agg(
            소분류수=("소분류", "nunique"),
            Pearson_평균=("Pearson", "mean"),
            Pearson_중앙값=("Pearson", "median"),
            Pearson_양수소분류수=("Pearson", lambda x: (x > 0).sum()),
            Spearman_평균=("Spearman", "mean"),
            Spearman_중앙값=("Spearman", "median"),
            Spearman_양수소분류수=("Spearman", lambda x: (x > 0).sum()),
        )
    )

    label_map = {
        "정부최근접접근성_음거리": "정부 최근접(-거리)",
        "서비스권역_인구비중": "인구비중",
        "SFCA_선호미반영": "선호 미반영 SFCA",
        "H3SFCA_선호반영": "선호 반영 H3SFCA",
    }
    corr["지표라벨"] = corr["지표명"].map(label_map)

    pearson_table = (
        corr.pivot_table(index="소분류", columns="지표라벨", values="Pearson", aggfunc="first")
        .reset_index()
    )
    spearman_table = (
        corr.pivot_table(index="소분류", columns="지표라벨", values="Spearman", aggfunc="first")
        .reset_index()
    )
    column_order = ["소분류", "선호 반영 H3SFCA", "선호 미반영 SFCA", "인구비중", "정부 최근접(-거리)"]
    pearson_table = pearson_table[[column for column in column_order if column in pearson_table.columns]]
    spearman_table = spearman_table[[column for column in column_order if column in spearman_table.columns]]
    return corr.drop(columns=["지표라벨"]), summary, pearson_table, spearman_table


def label_bar(axis, value, y_position):
    offset = 0.02
    if pd.isna(value):
        return
    if value >= 0:
        axis.text(value + offset, y_position, f"{value:.2f}", va="center", ha="left", fontsize=15, color=STYLE["text"])
    else:
        axis.text(value - offset, y_position, f"{value:.2f}", va="center", ha="right", fontsize=15, color=STYLE["text"])


def draw_split_axis(axis, data, focus_col, subtitle, xlim):
    y_positions = np.arange(len(data))
    bar_height = 0.36
    axis.barh(y_positions + bar_height / 2, data[focus_col], height=bar_height, color=STYLE["orange"])
    axis.barh(y_positions - bar_height / 2, data["정부 최근접(-거리)"], height=bar_height, color=STYLE["comparison"])

    for y_position, row in zip(y_positions, data.to_dict("records")):
        label_bar(axis, row[focus_col], y_position + bar_height / 2)
        label_bar(axis, row["정부 최근접(-거리)"], y_position - bar_height / 2)

    axis.set_yticks(y_positions)
    axis.set_yticklabels(data["소분류"], fontsize=18, color=STYLE["text"])
    axis.invert_yaxis()
    axis.axvline(0, color=STYLE["axis"], linewidth=1.1)
    axis.set_xlim(xlim)
    axis.set_title(subtitle, fontsize=22, fontweight="bold", color=STYLE["text"], pad=18)
    axis.grid(False)
    for spine in ["top", "right", "left"]:
        axis.spines[spine].set_visible(False)
    axis.spines["bottom"].set_color(STYLE["axis"])
    axis.spines["bottom"].set_linewidth(1)
    axis.tick_params(axis="x", colors=STYLE["tick"], labelsize=13)
    axis.tick_params(axis="y", length=0)
    axis.set_facecolor(STYLE["background"])


def draw_split_comparison(table: pd.DataFrame, method: str, focus_col: str, focus_label: str, filename: str):
    data = table[["소분류", focus_col, "정부 최근접(-거리)"]].replace([np.inf, -np.inf], np.nan).dropna().copy()
    data[focus_col] = pd.to_numeric(data[focus_col], errors="coerce")
    data["정부 최근접(-거리)"] = pd.to_numeric(data["정부 최근접(-거리)"], errors="coerce")
    data = data.dropna(subset=[focus_col, "정부 최근접(-거리)"])

    focus_higher = data.loc[data[focus_col] >= data["정부 최근접(-거리)"]].sort_values(focus_col, ascending=False)
    government_higher = data.loc[data[focus_col] < data["정부 최근접(-거리)"]].sort_values("정부 최근접(-거리)", ascending=False)

    max_abs = data[[focus_col, "정부 최근접(-거리)"]].abs().max().max()
    xlim = (-max(0.45, max_abs + 0.10), max(0.45, max_abs + 0.10))
    height = max(5.5, 0.54 * max(len(focus_higher), len(government_higher)) + 2.4)

    fig, axes = plt.subplots(1, 2, figsize=(18, height), sharex=True)
    fig.patch.set_facecolor(STYLE["background"])

    draw_split_axis(axes[0], focus_higher, focus_col, f"{focus_label}가 더 높은 소분류", xlim)
    draw_split_axis(axes[1], government_higher, focus_col, "최근접 접근성이 더 높은 소분류", xlim)

    legend_items = [
        Patch(facecolor=STYLE["orange"], label=focus_label),
        Patch(facecolor=STYLE["comparison"], label="정부 최근접(-거리)"),
    ]
    fig.legend(handles=legend_items, loc="upper center", bbox_to_anchor=(0.5, 0.935), ncol=2, frameon=False, fontsize=16)
    fig.suptitle(f"소분류별 {method} 상관계수 비교 - 25km 재계산", fontsize=30, fontweight="bold", color=STYLE["text"], y=0.99)
    plt.subplots_adjust(top=0.82, wspace=0.34)
    out_path = IMAGE_PATH / filename
    plt.savefig(out_path, dpi=220, bbox_inches="tight", facecolor=STYLE["background"])
    plt.close(fig)
    print("저장:", out_path)


def main():
    selected = load_selected_subcategories()

    seoul_grid_base = pd.read_csv(
        SEOUL_GRID_POP_PATH,
        encoding="utf-8-sig",
        usecols=["GRID_CD", "시군구", "행정동", "중심점_x", "중심점_y", "추정_인구수", "문화누리대상자_추정_인구수"],
    ).drop_duplicates("GRID_CD").copy()
    for col in ["추정_인구수", "문화누리대상자_추정_인구수"]:
        seoul_grid_base[col] = pd.to_numeric(seoul_grid_base[col], errors="coerce").fillna(0)

    grid_all = pd.read_parquet(
        GRID_COMPETITION_PATH,
        columns=["GRID_CD", "시도", "서울여부", "시군구", "행정동", "중심점_x", "중심점_y", "추정_인구수"],
    )
    grid_all["추정_인구수"] = pd.to_numeric(grid_all["추정_인구수"], errors="coerce").fillna(0)
    grid_all["서울여부"] = grid_all["서울여부"].astype(bool)

    usage = load_usage(selected, seoul_grid_base)
    store_25km = load_store_25km(selected)
    public_gu = calculate_public_nearest(selected, store_25km, seoul_grid_base)
    sfca_grid, h3_grid = calculate_sfca_h3(store_25km, selected, grid_all)

    sfca_gu = aggregate_grid_access(seoul_grid_base, selected, sfca_grid, "SFCA_선호미반영", "SFCA_선호미반영")
    h3_gu = aggregate_grid_access(seoul_grid_base, selected, h3_grid, "H3SFCA_선호반영", "H3SFCA_선호반영")
    h3_mode = h3_grid[["분석그룹", "중분류", "소분류", "H3SFCA_처리방식"]].drop_duplicates()

    indicator_wide = usage[
        ["시군구", "분석그룹", "중분류", "소분류", "이용건수", "대상자1인당_이용건수", "대상자천명당_이용건수", "구별_문화누리대상자추정인구"]
    ].copy()
    indicator_wide = indicator_wide.merge(
        public_gu[[
            "시군구", "분석그룹", "중분류", "소분류", "정부최근접_가중평균거리_m",
            "서비스권역_인구비중", "시설수", "시설수_서울", "시설수_외부25km", "거리결측격자수",
        ]],
        on=["시군구", "분석그룹", "중분류", "소분류"],
        how="left",
    )
    indicator_wide = indicator_wide.merge(
        sfca_gu[["시군구", "분석그룹", "중분류", "소분류", "SFCA_선호미반영"]],
        on=["시군구", "분석그룹", "중분류", "소분류"],
        how="left",
    )
    indicator_wide = indicator_wide.merge(
        h3_gu[["시군구", "분석그룹", "중분류", "소분류", "H3SFCA_선호반영"]],
        on=["시군구", "분석그룹", "중분류", "소분류"],
        how="left",
    )
    indicator_wide = indicator_wide.merge(h3_mode, on=["분석그룹", "중분류", "소분류"], how="left")
    indicator_wide["정부최근접접근성_음거리"] = -indicator_wide["정부최근접_가중평균거리_m"]

    indicator_long = indicator_wide.melt(
        id_vars=[
            "시군구", "분석그룹", "중분류", "소분류", "이용건수", "대상자1인당_이용건수",
            "대상자천명당_이용건수", "구별_문화누리대상자추정인구", "시설수", "시설수_서울", "시설수_외부25km",
        ],
        value_vars=["정부최근접접근성_음거리", "서비스권역_인구비중", "SFCA_선호미반영", "H3SFCA_선호반영"],
        var_name="지표명",
        value_name="지표값",
    )

    corr, summary, pearson_table, spearman_table = build_correlation_outputs(indicator_wide, indicator_long)

    unit_meta = (
        indicator_wide
        .groupby(["분석그룹", "중분류", "소분류"], as_index=False)
        .agg(
            시설수=("시설수", "max"),
            시설수_서울=("시설수_서울", "max"),
            시설수_외부25km=("시설수_외부25km", "max"),
            이용건수합=("이용건수", "sum"),
            대상자천명당_이용건수_평균=("대상자천명당_이용건수", "mean"),
            거리결측격자수합=("거리결측격자수", "sum"),
        )
    )

    suffix = "_25km_recalc"
    indicator_wide.to_csv(OUTPUT_PATH / f"subcategory_access_usage_gu_wide_2025{suffix}.csv", index=False, encoding="utf-8-sig")
    indicator_long.to_csv(OUTPUT_PATH / f"subcategory_access_usage_gu_long_2025{suffix}.csv", index=False, encoding="utf-8-sig")
    unit_meta.to_csv(OUTPUT_PATH / f"subcategory_access_usage_unit_meta_2025{suffix}.csv", index=False, encoding="utf-8-sig")
    corr.to_csv(OUTPUT_PATH / f"subcategory_access_usage_correlation_2025{suffix}.csv", index=False, encoding="utf-8-sig")
    summary.to_csv(OUTPUT_PATH / f"subcategory_access_usage_correlation_summary_2025{suffix}.csv", index=False, encoding="utf-8-sig")
    pearson_table.to_csv(OUTPUT_PATH / f"subcategory_access_usage_pearson_table_2025{suffix}.csv", index=False, encoding="utf-8-sig")
    spearman_table.to_csv(OUTPUT_PATH / f"subcategory_access_usage_spearman_table_2025{suffix}.csv", index=False, encoding="utf-8-sig")

    configure_project_font()
    draw_split_comparison(
        pearson_table,
        "Pearson",
        "선호 반영 H3SFCA",
        "선호 반영 H3SFCA",
        f"subcategory_pearson_government_vs_h3sfca_split_2025{suffix}.png",
    )
    draw_split_comparison(
        spearman_table,
        "Spearman",
        "선호 반영 H3SFCA",
        "선호 반영 H3SFCA",
        f"subcategory_spearman_government_vs_h3sfca_split_2025{suffix}.png",
    )
    draw_split_comparison(
        pearson_table,
        "Pearson",
        "선호 미반영 SFCA",
        "선호 미반영 SFCA",
        f"subcategory_pearson_government_vs_sfca_split_2025{suffix}.png",
    )
    draw_split_comparison(
        spearman_table,
        "Spearman",
        "선호 미반영 SFCA",
        "선호 미반영 SFCA",
        f"subcategory_spearman_government_vs_sfca_split_2025{suffix}.png",
    )

    print("저장 완료")
    print("wide:", indicator_wide.shape)
    print("long:", indicator_long.shape)
    print("unit_meta:", unit_meta.shape)
    print("corr:", corr.shape)
    print(unit_meta[unit_meta["소분류"].eq("온천")].to_string(index=False))


if __name__ == "__main__":
    main()


## 13-1. 중분류 Pearson 상관 비교: H3SFCA vs 최근접 접근
- H3SFCA 중분류 접근성은 25km 경쟁수요/경쟁공급 기준 산출물을 사용한다.
- 최근접 접근은 public_access_index_25km의 정부 최근접 접근성 중분류 시군구 산출물을 사용한다.
- 그래프 디자인은 아래 소분류 split 그래프와 같은 설정을 사용한다.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.patches import Patch
import numpy as np
import pandas as pd


PROJECT_PATH = Path(r"C:\project\oracle_mnc_project")
EDA_PATH = PROJECT_PATH / "notebooks" / "eda"
IMAGE_PATH = EDA_PATH / "IMAGE"
ACCESS_OUTPUT_PATH = PROJECT_PATH / "notebooks" / "access" / "OUTPUT"
H3_PATH = ACCESS_OUTPUT_PATH / "h3sfca"
PUBLIC_25KM_PATH = ACCESS_OUTPUT_PATH / "public_access_index_25km"
MNC_CARD_PATH = PROJECT_PATH / "data" / "raw" / "mnc_card" / "mnc_seoul_usage_issuance_2021_2025.xlsx"
IMAGE_PATH.mkdir(parents=True, exist_ok=True)

STYLE = {
    "background": "#fbf6ef",
    "text": "#241f1c",
    "axis": "#d9d3cb",
    "tick": "#766f67",
    "comparison": "#c7c2bc",
    "orange": "#f26b30",
    "teal": "#009688",
}

MIDDLE_CATEGORY_MAP = {
    "도서": ["도서"],
    "음악": ["음악"],
    "영상": ["영화", "TV"],
    "공연": ["공연"],
    "미술": ["전시", "공예", "사진관"],
    "문화체험": ["문화체험", "직업체험", "문화일반"],
    "관광지": ["관광명소", "휴양림캠핑장", "동식물원", "온천", "체험관광", "테마파크"],
    "스포츠관람": ["스포츠관람"],
    "체육용품": ["체육용품"],
    "체육시설": ["체육시설"],
}


def configure_project_font():
    font_source = Path(r"C:\Windows\Fonts\NotoSansKR-VF.ttf")
    if font_source.exists():
        try:
            import tempfile
            from fontTools.ttLib import TTFont
            from fontTools.varLib import instancer

            for weight in [400, 900]:
                font_path = Path(tempfile.gettempdir()) / f"NotoSansKR-{weight}.ttf"
                if not font_path.exists():
                    font = instancer.instantiateVariableFont(TTFont(str(font_source)), {"wght": weight})
                    font.save(str(font_path))
                font_manager.fontManager.addfont(str(font_path))

            plt.rcParams["font.family"] = "Noto Sans KR"
            plt.rcParams["axes.unicode_minus"] = False
            return
        except Exception:
            pass

    installed_fonts = {font.name for font in font_manager.fontManager.ttflist}
    for font_name in ["Noto Sans KR", "Malgun Gothic", "Apple SD Gothic Neo", "AppleGothic", "DejaVu Sans"]:
        if font_name in installed_fonts:
            plt.rcParams["font.family"] = font_name
            break
    else:
        plt.rcParams["font.family"] = "DejaVu Sans"
    plt.rcParams["axes.unicode_minus"] = False


def normalize_name(value):
    value = str(value).strip()
    for token in ["\n", " ", "/", "ㆍ", "·", "(", ")"]:
        value = value.replace(token, "")
    return value


def clean_number(series):
    return pd.to_numeric(series.astype(str).str.replace(",", "", regex=False).str.strip(), errors="coerce")


def weighted_mean(value, weight):
    value = pd.to_numeric(value, errors="coerce")
    weight = pd.to_numeric(weight, errors="coerce").fillna(0)
    valid = value.notna()
    if valid.sum() == 0:
        return np.nan
    value = value[valid]
    weight = weight[valid]
    if weight.sum() > 0:
        return np.average(value, weights=weight)
    return value.mean()


def safe_corr(data, x_col, y_col):
    temp = data[[x_col, y_col]].replace([np.inf, -np.inf], np.nan).dropna()
    if len(temp) < 3 or temp[x_col].nunique() < 2 or temp[y_col].nunique() < 2:
        return np.nan
    return temp[x_col].corr(temp[y_col], method="pearson")


def build_middle_usage():
    population = pd.read_csv(
        H3_PATH / "h3sfca_격자_중분류_접근성.csv",
        encoding="utf-8-sig",
        usecols=["GRID_CD", "시군구", "문화누리대상자_추정_인구수"],
    ).drop_duplicates("GRID_CD")
    population["문화누리대상자_추정_인구수"] = pd.to_numeric(
        population["문화누리대상자_추정_인구수"],
        errors="coerce",
    ).fillna(0)
    gu_population = (
        population.groupby("시군구", as_index=False)
        .agg(구별_문화누리대상자추정인구=("문화누리대상자_추정_인구수", "sum"))
    )

    raw = pd.read_excel(MNC_CARD_PATH, sheet_name="2025")
    raw["광역"] = raw["광역"].astype(str).str.strip()
    raw["기초"] = raw["기초"].astype(str).str.strip()
    raw = raw[raw["광역"].eq("서울")].rename(columns={"기초": "시군구"}).copy()

    count_columns = [column for column in raw.columns if str(column).endswith("(건)")]
    lookup = {normalize_name(str(column).replace("(건)", "")): column for column in count_columns}

    frames = []
    for middle_category, source_names in MIDDLE_CATEGORY_MAP.items():
        temp = raw[["시군구"]].copy()
        temp["이용건수"] = 0.0
        for source_name in source_names:
            column = lookup.get(normalize_name(source_name))
            if column is not None:
                temp["이용건수"] = temp["이용건수"] + clean_number(raw[column]).fillna(0)
        temp["중분류"] = middle_category
        frames.append(temp)

    usage = pd.concat(frames, ignore_index=True).merge(gu_population, on="시군구", how="left")
    usage["대상자1인당_이용건수"] = np.where(
        usage["구별_문화누리대상자추정인구"] > 0,
        usage["이용건수"] / usage["구별_문화누리대상자추정인구"],
        np.nan,
    )
    return usage


def build_middle_correlation_table():
    middle_categories = list(MIDDLE_CATEGORY_MAP)

    h3_grid = pd.read_csv(
        H3_PATH / "h3sfca_격자_중분류_접근성.csv",
        encoding="utf-8-sig",
        usecols=["시군구", "중분류", "접근성지수", "문화누리대상자_추정_인구수"],
    )
    h3_grid = h3_grid[h3_grid["중분류"].isin(middle_categories)].copy()
    h3_grid["접근성지수"] = pd.to_numeric(h3_grid["접근성지수"], errors="coerce")
    h3_grid["문화누리대상자_추정_인구수"] = pd.to_numeric(
        h3_grid["문화누리대상자_추정_인구수"],
        errors="coerce",
    ).fillna(0)

    h3_gu_rows = []
    for (gu, middle_category), group in h3_grid.groupby(["시군구", "중분류"], sort=False):
        h3_gu_rows.append(
            {
                "시군구": gu,
                "중분류": middle_category,
                "선호 반영 H3SFCA": weighted_mean(group["접근성지수"], group["문화누리대상자_추정_인구수"]),
            }
        )
    h3_gu = pd.DataFrame(h3_gu_rows)

    nearest = pd.read_csv(
        PUBLIC_25KM_PATH / "공공기관식_최근접접근성_서울시군구_중분류별.csv",
        encoding="utf-8-sig",
    )
    nearest = nearest[nearest["중분류"].isin(middle_categories)].copy()
    nearest["정부 최근접(-거리)"] = -pd.to_numeric(
        nearest["문화누리대상자_가중평균_접근거리_m"],
        errors="coerce",
    )
    nearest = nearest[["시군구", "중분류", "정부 최근접(-거리)"]]

    usage = build_middle_usage()
    wide = usage.merge(h3_gu, on=["시군구", "중분류"], how="left").merge(nearest, on=["시군구", "중분류"], how="left")

    rows = []
    for middle_category, group in wide.groupby("중분류", sort=False):
        rows.append(
            {
                "중분류": middle_category,
                "선호 반영 H3SFCA": safe_corr(group, "선호 반영 H3SFCA", "대상자1인당_이용건수"),
                "정부 최근접(-거리)": safe_corr(group, "정부 최근접(-거리)", "대상자1인당_이용건수"),
            }
        )
    return pd.DataFrame(rows)


def split_by_stronger_indicator(table, focus_col):
    data = table[["중분류", focus_col, "정부 최근접(-거리)"]].replace([np.inf, -np.inf], np.nan).dropna().copy()
    data[focus_col] = pd.to_numeric(data[focus_col], errors="coerce")
    data["정부 최근접(-거리)"] = pd.to_numeric(data["정부 최근접(-거리)"], errors="coerce")
    data = data.dropna(subset=[focus_col, "정부 최근접(-거리)"])

    focus_higher = data.loc[data[focus_col] >= data["정부 최근접(-거리)"]].sort_values(
        focus_col,
        ascending=False,
    )
    government_higher = data.loc[data[focus_col] < data["정부 최근접(-거리)"]].sort_values(
        "정부 최근접(-거리)",
        ascending=False,
    )
    return focus_higher, government_higher


def label_bar(axis, value, y_position):
    offset = 0.02
    if pd.isna(value):
        return
    if value >= 0:
        axis.text(
            value + offset,
            y_position,
            f"{value:.2f}",
            va="center",
            ha="left",
            fontsize=15,
            color=STYLE["text"],
        )
    else:
        axis.text(
            value - offset,
            y_position,
            f"{value:.2f}",
            va="center",
            ha="right",
            fontsize=15,
            color=STYLE["text"],
        )


def draw_split_axis(axis, data, focus_col, subtitle, xlim):
    y_positions = np.arange(len(data))
    bar_height = 0.36

    axis.barh(
        y_positions - bar_height / 2,
        data["정부 최근접(-거리)"],
        height=bar_height,
        color=STYLE["comparison"],
        edgecolor="none",
    )
    axis.barh(
        y_positions + bar_height / 2,
        data[focus_col],
        height=bar_height,
        color=STYLE["orange"],
        edgecolor="none",
    )

    for y_position, (_, row) in zip(y_positions, data.iterrows()):
        label_bar(axis, row["정부 최근접(-거리)"], y_position - bar_height / 2)
        label_bar(axis, row[focus_col], y_position + bar_height / 2)

    axis.axvline(0, color=STYLE["axis"], linewidth=1.35, zorder=0)
    axis.set_xlim(xlim)
    axis.set_yticks(y_positions)
    axis.set_yticklabels(data["중분류"], fontsize=18, color=STYLE["text"])
    axis.invert_yaxis()
    axis.set_title(subtitle, fontsize=23, fontweight=900, color=STYLE["text"], pad=24)
    axis.grid(False)
    axis.tick_params(axis="x", labelsize=13, colors=STYLE["tick"])
    axis.tick_params(axis="y", length=0)

    for spine in ["top", "right", "left"]:
        axis.spines[spine].set_visible(False)
    axis.spines["bottom"].set_color(STYLE["axis"])
    axis.set_facecolor(STYLE["background"])


def calculate_shared_xlim(*frames, columns):
    values = []
    for frame in frames:
        for column in columns:
            values.extend(frame[column].dropna().astype(float).tolist())
    if not values:
        return (-0.45, 0.8)
    lower = min(-0.45, np.floor((min(values) - 0.05) * 10) / 10)
    upper = max(0.8, np.ceil((max(values) + 0.05) * 10) / 10)
    return (lower, upper)


def plot_middle_government_vs_h3sfca_split():
    configure_project_font()
    focus_col = "선호 반영 H3SFCA"
    table = build_middle_correlation_table()
    focus_higher, government_higher = split_by_stronger_indicator(table, focus_col)
    xlim = calculate_shared_xlim(
        focus_higher,
        government_higher,
        columns=[focus_col, "정부 최근접(-거리)"],
    )

    fig, axes = plt.subplots(1, 2, figsize=(22, 10.5), dpi=180)
    fig.patch.set_facecolor(STYLE["background"])

    draw_split_axis(axes[0], focus_higher, focus_col, "H3FCA > 최근접 접근", xlim)
    draw_split_axis(axes[1], government_higher, focus_col, "H3SFCA < 최근접 접근", xlim)

    legend_handles = [
        Patch(facecolor=STYLE["orange"], edgecolor="none", label="선호 반영 H3SFCA"),
        Patch(facecolor=STYLE["comparison"], edgecolor="none", label="정부 최근접(-거리)"),
    ]
    fig.suptitle(
        "Pearson 상관 비교: H3FCA vs 최근접 접근",
        fontsize=35,
        fontweight=900,
        color=STYLE["text"],
        y=0.985,
    )
    fig.legend(
        handles=legend_handles,
        loc="upper center",
        bbox_to_anchor=(0.5, 0.925),
        ncol=2,
        frameon=False,
        fontsize=16,
        handlelength=1.8,
        columnspacing=2.6,
    )
    fig.tight_layout(rect=(0, 0, 1, 0.875), w_pad=5.0)

    output_path = IMAGE_PATH / "middle_category_pearson_government_vs_h3sfca_split_2025_25km.png"
    fig.savefig(output_path, dpi=180, bbox_inches="tight", facecolor=STYLE["background"])
    plt.close(fig)
    return output_path


middle_graph_path = plot_middle_government_vs_h3sfca_split()
print(middle_graph_path)


## 14. 원본 split 그래프 디자인으로 25km 재계산 그래프 재출력
- 25km 재계산 값은 유지한다.
- 원본 split 그래프의 캔버스, x축 범위, 제목/범례 위치, 막대 두께, 색상, 수치 표기 방식을 그대로 사용한다.


In [ ]:
from pathlib import Path

import nbformat
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.patches import Patch
import numpy as np
import pandas as pd


PROJECT_PATH = Path(r"C:\project\oracle_mnc_project")
EDA_PATH = PROJECT_PATH / "notebooks" / "eda"
OUTPUT_PATH = EDA_PATH / "OUTPUT"
IMAGE_PATH = EDA_PATH / "IMAGE"
NOTEBOOK_PATH = EDA_PATH / "02_sub_cat_correlation.ipynb"
IMAGE_PATH.mkdir(parents=True, exist_ok=True)

STYLE = {
    "background": "#fbf6ef",
    "text": "#241f1c",
    "axis": "#d9d3cb",
    "tick": "#766f67",
    "comparison": "#c7c2bc",
    "orange": "#f26b30",
    "teal": "#009688",
}

COMPARE_COL = "정부 최근접(-거리)"
FOCUS_COLUMNS = {
    "h3sfca": {
        "column": "선호 반영 H3SFCA",
        "legend": "선호 반영 H3SFCA",
        "left_title": "H3FCA > 최근접 접근",
        "right_title": "H3SFCA < 최근접 접근",
    },
    "sfca": {
        "column": "선호 미반영 SFCA",
        "legend": "선호 미반영 SFCA",
        "left_title": "SFCA가 더 높은 소분류",
    },
}


def configure_project_font():
    font_source = Path(r"C:\Windows\Fonts\NotoSansKR-VF.ttf")
    if font_source.exists():
        try:
            import tempfile
            from fontTools.ttLib import TTFont
            from fontTools.varLib import instancer

            for weight in [400, 900]:
                font_path = Path(tempfile.gettempdir()) / f"NotoSansKR-{weight}.ttf"
                if not font_path.exists():
                    font = instancer.instantiateVariableFont(TTFont(str(font_source)), {"wght": weight})
                    font.save(str(font_path))
                font_manager.fontManager.addfont(str(font_path))

            plt.rcParams["font.family"] = "Noto Sans KR"
            plt.rcParams["axes.unicode_minus"] = False
            return
        except Exception:
            pass

    installed_fonts = {font.name for font in font_manager.fontManager.ttflist}
    for font_name in ["Noto Sans KR", "Malgun Gothic", "Apple SD Gothic Neo", "AppleGothic", "DejaVu Sans"]:
        if font_name in installed_fonts:
            plt.rcParams["font.family"] = font_name
            break
    else:
        plt.rcParams["font.family"] = "DejaVu Sans"
    plt.rcParams["axes.unicode_minus"] = False


def read_recalc_correlation_table(method):
    method = method.lower()
    if method not in {"pearson", "spearman"}:
        raise ValueError("method는 'pearson' 또는 'spearman'만 지원합니다.")
    path = OUTPUT_PATH / f"subcategory_access_usage_{method}_table_2025_25km_recalc.csv"
    return pd.read_csv(path, encoding="utf-8-sig")


def split_by_stronger_indicator(table, focus_col):
    required = ["소분류", focus_col, COMPARE_COL]
    missing = [column for column in required if column not in table.columns]
    if missing:
        raise ValueError(f"필수 열이 없습니다: {missing}")

    data = table[required].replace([np.inf, -np.inf], np.nan).dropna().copy()
    data[focus_col] = pd.to_numeric(data[focus_col], errors="coerce")
    data[COMPARE_COL] = pd.to_numeric(data[COMPARE_COL], errors="coerce")
    data = data.dropna(subset=[focus_col, COMPARE_COL])

    focus_higher = data.loc[data[focus_col] >= data[COMPARE_COL]].sort_values(
        focus_col,
        ascending=False,
    )
    government_higher = data.loc[data[focus_col] < data[COMPARE_COL]].sort_values(
        COMPARE_COL,
        ascending=False,
    )
    return focus_higher, government_higher


def label_bar(axis, value, y_position):
    offset = 0.02
    if pd.isna(value):
        return
    if value >= 0:
        axis.text(
            value + offset,
            y_position,
            f"{value:.2f}",
            va="center",
            ha="left",
            fontsize=15,
            color=STYLE["text"],
        )
    else:
        axis.text(
            value - offset,
            y_position,
            f"{value:.2f}",
            va="center",
            ha="right",
            fontsize=15,
            color=STYLE["text"],
        )


def draw_split_axis(axis, data, focus_col, subtitle, xlim):
    y_positions = np.arange(len(data))
    bar_height = 0.36

    axis.barh(
        y_positions - bar_height / 2,
        data[COMPARE_COL],
        height=bar_height,
        color=STYLE["comparison"],
        edgecolor="none",
    )
    axis.barh(
        y_positions + bar_height / 2,
        data[focus_col],
        height=bar_height,
        color=STYLE["orange"],
        edgecolor="none",
    )

    for y_position, (_, row) in zip(y_positions, data.iterrows()):
        label_bar(axis, row[COMPARE_COL], y_position - bar_height / 2)
        label_bar(axis, row[focus_col], y_position + bar_height / 2)

    axis.axvline(0, color=STYLE["axis"], linewidth=1.35, zorder=0)
    axis.set_xlim(xlim)
    axis.set_yticks(y_positions)
    axis.set_yticklabels(data["소분류"], fontsize=18, color=STYLE["text"])
    axis.invert_yaxis()
    axis.set_title(subtitle, fontsize=23, fontweight=900, color=STYLE["text"], pad=24)
    axis.grid(False)
    axis.tick_params(axis="x", labelsize=13, colors=STYLE["tick"])
    axis.tick_params(axis="y", length=0)

    for spine in ["top", "right", "left"]:
        axis.spines[spine].set_visible(False)
    axis.spines["bottom"].set_color(STYLE["axis"])
    axis.set_facecolor(STYLE["background"])


def calculate_shared_xlim(*frames, columns):
    values = []
    for frame in frames:
        for column in columns:
            values.extend(frame[column].dropna().astype(float).tolist())
    if not values:
        return (-0.45, 0.8)
    lower = min(-0.45, np.floor((min(values) - 0.05) * 10) / 10)
    upper = max(0.8, np.ceil((max(values) + 0.05) * 10) / 10)
    return (lower, upper)


def plot_recalc_government_vs_focus_split(method, focus_key):
    configure_project_font()
    focus = FOCUS_COLUMNS[focus_key]
    focus_col = focus["column"]
    table = read_recalc_correlation_table(method)
    focus_higher, government_higher = split_by_stronger_indicator(table, focus_col)
    xlim = calculate_shared_xlim(
        focus_higher,
        government_higher,
        columns=[focus_col, COMPARE_COL],
    )

    metric_title = "Pearson" if method.lower() == "pearson" else "Spearman"
    fig, axes = plt.subplots(1, 2, figsize=(22, 10.5), dpi=180)
    fig.patch.set_facecolor(STYLE["background"])

    draw_split_axis(axes[0], focus_higher, focus_col, focus["left_title"], xlim)
    draw_split_axis(axes[1], government_higher, focus_col, focus.get("right_title", "최근접 접근성이 더 높은 소분류"), xlim)

    legend_handles = [
        Patch(facecolor=STYLE["orange"], edgecolor="none", label=focus["legend"]),
        Patch(facecolor=STYLE["comparison"], edgecolor="none", label=COMPARE_COL),
    ]
    main_title = (
        "Pearson 상관 비교: H3FCA vs 최근접 접근"
        if method.lower() == "pearson" and focus_key == "h3sfca"
        else f"접근성 지수 - 문화누리 이용건수(1인당) {metric_title} 상관계수"
    )
    fig.suptitle(
        main_title,
        fontsize=35,
        fontweight=900,
        color=STYLE["text"],
        y=0.985,
    )
    fig.legend(
        handles=legend_handles,
        loc="upper center",
        bbox_to_anchor=(0.5, 0.925),
        ncol=2,
        frameon=False,
        fontsize=16,
        handlelength=1.8,
        columnspacing=2.6,
    )
    fig.tight_layout(rect=(0, 0, 1, 0.875), w_pad=5.0)

    output_path = IMAGE_PATH / f"subcategory_{method.lower()}_government_vs_{focus_key}_split_2025_25km_recalc.png"
    fig.savefig(output_path, dpi=180, bbox_inches="tight", facecolor=STYLE["background"])
    plt.close(fig)
    return output_path


def append_notebook_cell():
    marker = "## 14. 원본 split 그래프 디자인으로 25km 재계산 그래프 재출력"
    nb = nbformat.read(NOTEBOOK_PATH, as_version=4)
    nb.cells = [cell for cell in nb.cells if marker not in "".join(cell.get("source", ""))]

    script_source = Path(__file__).read_text(encoding="utf-8")
    script_source = script_source.replace("\nappend_notebook_cell()\n", "\n")
    script_source = script_source.replace("\nif __name__ == \"__main__\":\n    main()\n", "\nmain()\n")
    markdown = (
        f"{marker}\n"
        "- 25km 재계산 값은 유지한다.\n"
        "- 원본 split 그래프의 캔버스, x축 범위, 제목/범례 위치, 막대 두께, 색상, 수치 표기 방식을 그대로 사용한다.\n"
    )
    nb.cells.append(nbformat.v4.new_markdown_cell(markdown))
    nb.cells.append(nbformat.v4.new_code_cell(script_source))
    nbformat.write(nb, NOTEBOOK_PATH)


def main():
    generated = []
    for method_name in ["pearson", "spearman"]:
        for focus_key in ["h3sfca", "sfca"]:
            generated.append(plot_recalc_government_vs_focus_split(method_name, focus_key))

    print("원본 디자인으로 재생성한 25km split 그래프")
    for path in generated:
        print("-", path)


main()


## 15. Pearson 상관계수 차이 유의성 검정
- 그래프에 표시된 H3SFCA-이용건수 Pearson 상관계수와 정부 최근접-이용건수 Pearson 상관계수의 차이를 Williams test로 검정함.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
from IPython.display import display

BASE_PATH = Path().resolve()

if BASE_PATH.name == "eda":
    PROJECT_PATH = BASE_PATH.parents[1]
elif BASE_PATH.name == "notebooks":
    PROJECT_PATH = BASE_PATH.parent
elif (BASE_PATH / "analysis_table").exists():
    PROJECT_PATH = BASE_PATH
else:
    PROJECT_PATH = Path(r"C:\project\oracle_mnc_project")

OUTPUT_PATH = PROJECT_PATH / "notebooks" / "eda" / "OUTPUT"
wide = pd.read_csv(OUTPUT_PATH / "subcategory_access_usage_gu_wide_2025_25km_recalc.csv", encoding="utf-8-sig")

target_col = "대상자1인당_이용건수"
h3_col = "H3SFCA_선호반영"
gov_col = "정부최근접접근성_음거리"


def williams_test_shared_target(data: pd.DataFrame) -> float:
    temp = data[[target_col, h3_col, gov_col]].dropna()

    if len(temp) < 4 or any(temp[col].nunique() < 2 for col in [target_col, h3_col, gov_col]):
        return np.nan

    r_h3 = temp[target_col].corr(temp[h3_col])
    r_gov = temp[target_col].corr(temp[gov_col])
    r_between = temp[h3_col].corr(temp[gov_col])
    n = len(temp)

    determinant = 1 - r_h3**2 - r_gov**2 - r_between**2 + 2 * r_h3 * r_gov * r_between
    denominator = (
        2 * ((n - 1) / (n - 3)) * determinant
        + ((r_h3 + r_gov) ** 2 / 4) * (1 - r_between) ** 3
    )

    if denominator <= 0 or pd.isna(denominator):
        return np.nan

    t_stat = (r_h3 - r_gov) * np.sqrt((n - 1) * (1 + r_between) / denominator)
    return 2 * stats.t.sf(abs(t_stat), df=n - 3)


rows = []
for subcategory, group in wide.groupby("소분류", sort=False):
    p_value = williams_test_shared_target(group)
    is_significant = pd.notna(p_value) and p_value < 0.05

    rows.append(
        {
            "소분류": subcategory,
            "비교": "H3SFCA vs 정부 최근접",
            "p값": "" if pd.isna(p_value) else f"{p_value:.4f}",
            "유의수준": "0.05",
            "유의성": "유의함" if is_significant else "유의하지 않음",
        }
    )

significance_result = pd.DataFrame(rows)
significance_result = (
    significance_result.assign(
        _유의정렬=significance_result["유의성"].eq("유의함"),
        _p정렬=pd.to_numeric(significance_result["p값"], errors="coerce"),
    )
    .sort_values(["_유의정렬", "_p정렬", "소분류"], ascending=[False, True, True])
    .drop(columns=["_유의정렬", "_p정렬"])
)

display(significance_result)
